# **INSTRUCTIONS**

1. Make sure to ***Install Required Packages*** before running any code

2. For fresh data in the focus of the past 3 month run ***Scraping*** code before running the ***Model*** code
*   There are 3 seperate categories in the ***Scraping*** code are Anime, Movies, and Video Games
*   There will be **5-6 CSV files** automatically saved after running the code, named:

    * "anime_reddit.csv"
    * "video_game_reddit.csv"
    * "video_game_twitch.csv"
    * "movie_tmdb.csv"
    * "movie_reddit.csv"

* The same data is also saved to a Google Sheets named Database_overall with all 3 categories of data. This is the link to the Google Sheets: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit?usp=sharing

3. Run the ***Model*** code before running the ***Dashboard*** code.
 *   You will be asked for Google Autherization. Make sure to sign in with the same Google Account you are using in this Google Collab and give all access.

4. Lastly run the ***Dashboard*** code and click the link listed after "Running on public URL:"
* If any error occures make sure to run **ALL** code before running ***Dashboard*** code
<br>

 *NOTE: All code may take up to 30 minutes to run in total*
  
DOCUMENTATION: https://docs.google.com/document/d/1d2eOuKRUL-wLAf7bPpLRNj0J2d7zaNMthShT3kN3TME/edit?usp=sharing

## Installing Required Packages

In [ ]:
pip install scikit-learn

In [ ]:
pip install gradio plotly pandas

In [ ]:
pip install requests praw python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 6.2 MB/s eta 0:00:00


In [ ]:
pip install jikanpy praw pandas gspread google-auth google-colab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import requests
import time
import json
import os
from datetime import datetime, timedelta
import logging
import praw
import numpy as np
from collections import Counter, defaultdict
import re
from typing import List, Dict, Optional, Tuple
import gradio as gr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
from typing import Dict, List, Tuple, Any
from io import StringIO

# Google Sheets imports
try:
    import gspread
    from google.auth import default
    from google.colab import auth
except ImportError:
    print("Install Google packages: pip install gspread google-auth google-colab")

# Optional imports
try:
    from jikanpy import Jikan
except ImportError:
    print("Install jikanpy: pip install jikanpy")

try:
    import praw
except ImportError:
    print("Install praw: pip install praw")

# Reddit imports
try:
    import praw
except ImportError:
    print("Install praw: pip install praw")

from jikanpy import Jikan


## Scraping

In [ ]:
# Standard library imports
import logging
import time
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter

# Google Sheets imports
try:
    import gspread
    from google.auth import default
    from google.colab import auth
except ImportError:
    print("Install Google packages: pip install gspread google-auth google-colab")

# Optional imports
try:
    from jikanpy import Jikan
except ImportError:
    print("Install jikanpy: pip install jikanpy")

try:
    import praw
except ImportError:
    print("Install praw: pip install praw")

# Reddit imports
try:
    import praw
except ImportError:
    print("Install praw: pip install praw")

from jikanpy import Jikan


### Anime

In [ ]:
### Anime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class AnimeDataCollector:
    def __init__(self, sheet_id=None, worksheet_name=None):
        """Initialize the anime data collector"""
        self.jikan = None
        self.reddit = None

        # Google Sheets setup
        self.sheet_id = sheet_id
        self.worksheet_name = worksheet_name
        self.worksheet = None

        # Data storage
        self.anime_data = []
        self.reddit_data = []

        # Time filter (last 6 months for more comprehensive collection)
        self.six_months_ago = datetime.now() - timedelta(days=180)
        self.three_months_ago = datetime.now() - timedelta(days=90)

        # Initialize APIs
        self._setup_apis()
        self._setup_google_sheets()

    def _setup_google_sheets(self):
        """Setup Google Sheets authentication and access"""
        if self.sheet_id and self.worksheet_name:
            try:
                auth.authenticate_user()
                creds, _ = default()
                client = gspread.authorize(creds)
                sheet = client.open_by_key(self.sheet_id)
                self.worksheet = sheet.worksheet(self.worksheet_name)
                logger.info(f"Google Sheets initialized successfully - Worksheet: {self.worksheet_name}")
            except Exception as e:
                logger.error(f"Failed to initialize Google Sheets: {e}")

    def _setup_apis(self):
        """Setup API clients"""
        # Jikan API setup (MyAnimeList)
        try:
            self.jikan = Jikan()
            logger.info("Jikan API initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize Jikan API: {e}")
            logger.error("Make sure jikanpy is installed: pip install jikanpy")

        # Reddit API setup - CREDENTIALS INTEGRATED
        try:
            self.reddit = praw.Reddit(
                client_id="HA15MkoZrg4xng2Y7umGsA",
                client_secret="yOJTSrreZG8rruSLOEOxi4XkASS7bw",
                user_agent='anime_data_collector_v1.0'
            )
            logger.info("Reddit API initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize Reddit API: {e}")

    def collect_anime_data(self, pages=3):
        """Collect anime data from MyAnimeList via Jikan API with proper rate limiting"""
        logger.info("Starting MyAnimeList anime data collection...")
        logger.info("Note: Jikan API has strict rate limits (30 requests/minute, 2 requests/second)")

        if not self.jikan:
            logger.error("Jikan API not available - please install jikanpy: pip install jikanpy")
            return

        # Track API calls for rate limiting
        api_calls = 0
        max_calls_per_minute = 25  # Stay under 30/minute limit

        try:
            # 1. Get current season anime (most reliable endpoint)
            logger.info("Collecting current season anime...")
            current_year = datetime.now().year
            current_month = datetime.now().month

            # Determine current season
            if current_month in [12, 1, 2]:
                season = 'winter'
            elif current_month in [3, 4, 5]:
                season = 'spring'
            elif current_month in [6, 7, 8]:
                season = 'summer'
            else:
                season = 'fall'

            try:
                logger.info(f"Requesting {season} {current_year} anime...")
                time.sleep(2)  # Initial delay

                current_season_data = self.jikan.seasons(year=current_year, season=season)
                api_calls += 1

                if current_season_data and 'data' in current_season_data:
                    logger.info(f"Successfully found {len(current_season_data['data'])} anime in {season} {current_year}")

                    for anime in current_season_data['data'][:50]:  # Limit to first 50
                        self._process_anime(anime, source='current_season')
                else:
                    logger.warning("No data returned from current season API")

                time.sleep(3)  # Wait between major API calls

            except Exception as e:
                logger.error(f"Error collecting current season: {e}")
                logger.error(f"Exception type: {type(e).__name__}")

            # 2. Get top anime (simpler, more reliable)
            if api_calls < max_calls_per_minute:
                logger.info("Collecting top anime...")
                try:
                    logger.info("Requesting top anime (page 1)...")
                    time.sleep(3)

                    top_anime_data = self.jikan.top(type='anime', page=1)
                    api_calls += 1

                    if top_anime_data and 'data' in top_anime_data:
                        logger.info(f"Successfully found {len(top_anime_data['data'])} top anime")

                        for anime in top_anime_data['data'][:25]:  # Limit to first 25
                            if not any(a['mal_id'] == anime['mal_id'] for a in self.anime_data):
                                self._process_anime(anime, source='top_anime')
                    else:
                        logger.warning("No data returned from top anime API")

                    time.sleep(3)

                except Exception as e:
                    logger.error(f"Error collecting top anime: {e}")
                    logger.error(f"Exception type: {type(e).__name__}")

            # 3. Try one more simple call - currently airing
            if api_calls < max_calls_per_minute:
                logger.info("Collecting currently airing anime...")
                try:
                    logger.info("Requesting currently airing anime...")
                    time.sleep(3)

                    airing_data = self.jikan.top(type='anime', filter='airing', page=1)
                    api_calls += 1

                    if airing_data and 'data' in airing_data:
                        logger.info(f"Successfully found {len(airing_data['data'])} airing anime")

                        for anime in airing_data['data'][:25]:  # Limit to first 25
                            if not any(a['mal_id'] == anime['mal_id'] for a in self.anime_data):
                                self._process_anime(anime, source='airing')
                    else:
                        logger.warning("No data returned from airing anime API")

                except Exception as e:
                    logger.error(f"Error collecting airing anime: {e}")
                    logger.error(f"Exception type: {type(e).__name__}")

        except Exception as e:
            logger.error(f"Major error in anime data collection: {e}")
            logger.error(f"Exception type: {type(e).__name__}")

        logger.info(f"Total API calls made: {api_calls}")
        logger.info(f"Total anime collected from MyAnimeList: {len(self.anime_data)}")

        if len(self.anime_data) == 0:
            logger.error("No anime data collected! Possible issues:")
            logger.error("1. Rate limiting - try running again after a few minutes")
            logger.error("2. Jikan API may be temporarily down")
            logger.error("3. Check your internet connection")
            logger.error("4. Try: pip install --upgrade jikanpy")

    def _process_anime(self, anime, source='unknown'):
        """Process individual anime data"""
        try:
            # Get additional details if needed
            mal_id = anime.get('mal_id')

            anime_record = {
                'mal_id': mal_id,
                'title': anime.get('title', ''),
                'title_english': anime.get('title_english', ''),
                'title_japanese': anime.get('title_japanese', ''),
                'score': anime.get('score'),
                'scored_by': anime.get('scored_by'),
                'rank': anime.get('rank'),
                'popularity': anime.get('popularity'),
                'members': anime.get('members'),
                'favorites': anime.get('favorites'),
                'genres': ', '.join([genre.get('name', '') for genre in anime.get('genres', [])]),
                'themes': ', '.join([theme.get('name', '') for theme in anime.get('themes', [])]),
                'demographics': ', '.join([demo.get('name', '') for demo in anime.get('demographics', [])]),
                'studios': ', '.join([studio.get('name', '') for studio in anime.get('studios', [])]),
                'type': anime.get('type'),
                'episodes': anime.get('episodes'),
                'status': anime.get('status'),
                'aired_from': anime.get('aired', {}).get('from') if anime.get('aired') else None,
                'aired_to': anime.get('aired', {}).get('to') if anime.get('aired') else None,
                'season': anime.get('season'),
                'year': anime.get('year'),
                'synopsis': (anime.get('synopsis') or '')[:500],
                'background': (anime.get('background') or '')[:300],
                'rating': anime.get('rating'),
                'source': anime.get('source'),
                'duration': anime.get('duration'),
                'url': anime.get('url'),
                'image_url': anime.get('images', {}).get('jpg', {}).get('image_url') if anime.get('images') else '',
                'trailer_url': anime.get('trailer', {}).get('url') if anime.get('trailer') else '',
                'collection_source': source,
                'platform': 'anime',
                'collected_date': datetime.now().isoformat()
            }

            self.anime_data.append(anime_record)

        except Exception as e:
            logger.error(f"Error processing anime {anime.get('title', 'Unknown')}: {e}")

    def collect_reddit_data(self, subreddits=None, limit=200):
        """Collect anime-related posts from Reddit (last 3 months)"""
        logger.info("Starting Reddit anime data collection...")

        if not self.reddit:
            logger.error("Reddit API not available")
            return

        if not subreddits:
            subreddits = ['anime', 'AnimeSuggestions', 'anime_irl', 'Animemes',
                         'visualnovels', 'manga', 'LightNovels', 'WeebTV',
                         'MyAnimeList', 'animediscussion']

        three_months_timestamp = self.three_months_ago.timestamp()

        # Anime-related keywords for better filtering
        anime_keywords = ['anime', 'manga', 'otaku', 'shounen', 'seinen', 'shoujo', 'josei',
                         'isekai', 'mecha', 'slice of life', 'rom com', 'harem', 'ecchi',
                         'studio', 'seasonal', 'waifu', 'best girl', 'mal', 'myanimelist',
                         'crunchyroll', 'funimation', 'episode', 'season', 'opening', 'ending']

        for subreddit_name in subreddits:
            try:
                subreddit = self.reddit.subreddit(subreddit_name)
                posts_collected = 0

                logger.info(f"Collecting from r/{subreddit_name}...")

                for submission in subreddit.hot(limit=limit * 2):
                    if submission.created_utc >= three_months_timestamp:
                        # Check for anime-related content
                        full_text = f"{submission.title} {submission.selftext}".lower()
                        has_anime_content = any(keyword in full_text for keyword in anime_keywords)

                        # For anime subreddits, accept all posts; for others, filter by keywords
                        anime_subreddits = ['anime', 'AnimeSuggestions', 'anime_irl', 'Animemes', 'MyAnimeList']
                        if subreddit_name in anime_subreddits or has_anime_content:
                            reddit_record = {
                                'reddit_id': submission.id,
                                'subreddit': subreddit_name,
                                'title': submission.title,
                                'author': str(submission.author) if submission.author else '[deleted]',
                                'content': (submission.selftext or '')[:500],
                                'score': submission.score,
                                'upvote_ratio': submission.upvote_ratio,
                                'num_comments': submission.num_comments,
                                'created_utc': datetime.utcfromtimestamp(submission.created_utc).isoformat(),
                                'url': f"https://reddit.com{submission.permalink}",
                                'flair': submission.link_flair_text,
                                'is_nsfw': submission.over_18,
                                'is_spoiler': submission.spoiler,
                                'platform': 'reddit_anime',
                                'collected_date': datetime.now().isoformat()
                            }
                            self.reddit_data.append(reddit_record)
                            posts_collected += 1

                            if posts_collected >= limit:
                                break

                logger.info(f"Collected {posts_collected} posts from r/{subreddit_name}")
                time.sleep(2)  # Rate limiting

            except Exception as e:
                logger.error(f"Error collecting Reddit data from r/{subreddit_name}: {e}")

    def save_data_to_csv(self, filename_prefix="anime"):
        """Save collected data to CSV files with consistent 'anime' naming"""
        # Save MAL/anime data as "anime_mal.csv"
        if self.anime_data:
            anime_df = pd.DataFrame(self.anime_data)
            anime_filename = f"{filename_prefix}_mal.csv"
            anime_df.to_csv(anime_filename, index=False)
            logger.info(f"Saved {len(self.anime_data)} anime records to {anime_filename}")

        # Save Reddit data as "anime_reddit.csv"
        if self.reddit_data:
            reddit_df = pd.DataFrame(self.reddit_data)
            reddit_filename = f"{filename_prefix}_reddit.csv"
            reddit_df.to_csv(reddit_filename, index=False)
            logger.info(f"Saved {len(self.reddit_data)} Reddit records to {reddit_filename}")

    def upload_to_google_sheets(self):
        """Upload all collected data to Google Sheets"""
        if not self.worksheet:
            logger.error("Google Sheets not initialized")
            return

        try:
            all_data = []

            # Create comprehensive headers for all data types
            headers = [
                'Data_Type', 'ID', 'Title', 'Title_English', 'Title_Japanese',
                'Score', 'Scored_By', 'Rank', 'Popularity', 'Members', 'Favorites',
                'Genres', 'Themes', 'Demographics', 'Studios', 'Type',
                'Episodes', 'Status', 'Aired_From', 'Aired_To', 'Season', 'Year',
                'Synopsis', 'Background', 'Rating', 'Source', 'Duration',
                'URL', 'Image_URL', 'Trailer_URL', 'Collection_Source', 'Platform', 'Collected_Date',
                # Reddit specific fields
                'Subreddit', 'Author', 'Content', 'Upvote_Ratio', 'Num_Comments',
                'Flair', 'Is_NSFW', 'Is_Spoiler'
            ]

            all_data.append(headers)

            # Add anime data
            for anime in self.anime_data:
                row = [
                    'ANIME',
                    anime.get('mal_id', ''),
                    anime.get('title', ''),
                    anime.get('title_english', ''),
                    anime.get('title_japanese', ''),
                    anime.get('score', ''),
                    anime.get('scored_by', ''),
                    anime.get('rank', ''),
                    anime.get('popularity', ''),
                    anime.get('members', ''),
                    anime.get('favorites', ''),
                    anime.get('genres', ''),
                    anime.get('themes', ''),
                    anime.get('demographics', ''),
                    anime.get('studios', ''),
                    anime.get('type', ''),
                    anime.get('episodes', ''),
                    anime.get('status', ''),
                    anime.get('aired_from', ''),
                    anime.get('aired_to', ''),
                    anime.get('season', ''),
                    anime.get('year', ''),
                    anime.get('synopsis', ''),
                    anime.get('background', ''),
                    anime.get('rating', ''),
                    anime.get('source', ''),
                    anime.get('duration', ''),
                    anime.get('url', ''),
                    anime.get('image_url', ''),
                    anime.get('trailer_url', ''),
                    anime.get('collection_source', ''),
                    anime.get('platform', ''),
                    anime.get('collected_date', ''),
                    # Empty Reddit fields
                    '', '', '', '', '', '', '', ''
                ]
                all_data.append(row)

            # Add Reddit data
            for reddit in self.reddit_data:
                row = [
                    'REDDIT',
                    reddit.get('reddit_id', ''),
                    reddit.get('title', ''),
                    '', '', # Empty English/Japanese titles
                    reddit.get('score', ''),
                    '', '', '', '', '', # Empty anime-specific fields
                    '', '', '', '', 'Post', # Type = Post
                    '', '', '', '', '', '',
                    '', '', '', '', '',
                    reddit.get('url', ''),
                    '', '', # Empty image/trailer
                    '', # Empty collection source
                    reddit.get('platform', ''),
                    reddit.get('collected_date', ''),
                    # Reddit specific fields
                    reddit.get('subreddit', ''),
                    reddit.get('author', ''),
                    reddit.get('content', ''),
                    reddit.get('upvote_ratio', ''),
                    reddit.get('num_comments', ''),
                    reddit.get('flair', ''),
                    reddit.get('is_nsfw', ''),
                    reddit.get('is_spoiler', '')
                ]
                all_data.append(row)

            # Clear and update worksheet
            self.worksheet.clear()
            self.worksheet.update(all_data)

            logger.info(f"Uploaded {len(all_data)-1} records to Google Sheets")
            print(f"Your Google Sheet: https://docs.google.com/spreadsheets/d/{self.sheet_id}/edit")

        except Exception as e:
            logger.error(f"Error uploading to Google Sheets: {e}")

    def get_summary_stats(self):
        """Print summary statistics"""
        print("\n=== ANIME DATA COLLECTION SUMMARY ===")
        print(f"MyAnimeList records: {len(self.anime_data)}")
        print(f"Reddit records: {len(self.reddit_data)}")
        print(f"Total records: {len(self.anime_data) + len(self.reddit_data)}")

        if self.anime_data:
            # Show some stats
            genres = []
            sources = []

            for anime in self.anime_data:
                if anime.get('genres'):
                    genres.extend([g.strip() for g in anime['genres'].split(',')])
                if anime.get('collection_source'):
                    sources.append(anime['collection_source'])

            top_genres = Counter(genres).most_common(5)
            collection_sources = Counter(sources).most_common()

            print(f"Top 5 genres: {top_genres}")
            print(f"Collection sources: {collection_sources}")

            # Average score
            scores = [anime.get('score') for anime in self.anime_data if anime.get('score')]
            if scores:
                avg_score = sum(scores) / len(scores)
                print(f"Average anime score: {avg_score:.2f}")

        print(f"Data collection period: Last 3 months from {datetime.now().strftime('%Y-%m-%d')}")

def main():
    """Main function for anime data collection"""
    print("Starting Anime Data Collection for Database_overall...")

    # Your Google Sheets configuration
    SHEET_ID = '1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk'
    WORKSHEET_NAME = 'Anime'

    collector = AnimeDataCollector(sheet_id=SHEET_ID, worksheet_name=WORKSHEET_NAME)

    # Collect data
    collector.collect_anime_data(pages=3)
    collector.collect_reddit_data(limit=100)

    # Save and upload
    collector.save_data_to_csv()
    collector.upload_to_google_sheets()
    collector.get_summary_stats()

    print("Anime data collection complete!")
    print(f"Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit")

if __name__ == "__main__":
    print("ANIME DATA COLLECTOR - READY TO RUN!")
    print("="*60)
    print("Reddit API Credentials: CONFIGURED")
    print("MyAnimeList API: Using Jikan (Free)")
    print()
    print("REQUIRED PACKAGES:")
    print("   pip install jikanpy praw pandas gspread google-auth google-colab")
    print()
    print("GOOGLE SHEETS ACCESS:")
    print("   - Make sure you have access to the Google Sheet")
    print("   - Run in Google Colab for easy authentication")
    print()
    print("TO RUN:")
    print("   python anime_collector.py")
    print("   # OR uncomment main() call below")
    print("="*60)

    # Uncomment the next line to run the script
    main()

ANIME DATA COLLECTOR - READY TO RUN!
Reddit API Credentials: CONFIGURED
MyAnimeList API: Using Jikan (Free)

REQUIRED PACKAGES:
   pip install jikanpy praw pandas gspread google-auth google-colab

GOOGLE SHEETS ACCESS:
   - Make sure you have access to the Google Sheet
   - Run in Google Colab for easy authentication

TO RUN:
   python anime_collector.py
   # OR uncomment main() call below
Starting Anime Data Collection for Database_overall...


ERROR:__main__:Error collecting current season: 'Jikan' object has no attribute 'seasons'
ERROR:__main__:Exception type: AttributeError
ERROR:__main__:Error collecting top anime: HTTP 410 - status=410, type=BadResponseException, message=v3 has been discontinued. For more information visit https://bit.ly/jikan-v3-deprecation, upgrade=Please upgrade to v4: https://docs.api.jikan.moe/ for type=anime
ERROR:__main__:Exception type: APIException
ERROR:__main__:Error collecting airing anime: Jikan.top() got an unexpected keyword argument 'filter'
ERROR:__main__:Exception type: TypeError
ERROR:__main__:No anime data collected! Possible issues:
ERROR:__main__:1. Rate limiting - try running again after a few minutes
ERROR:__main__:2. Jikan API may be temporarily down
ERROR:__main__:3. Check your internet connection
ERROR:__main__:4. Try: pip install --upgrade jikanpy
It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_

Your Google Sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit

=== ANIME DATA COLLECTION SUMMARY ===
MyAnimeList records: 0
Reddit records: 635
Total records: 635
Data collection period: Last 3 months from 2025-09-16
Anime data collection complete!
Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit


### Movies

In [ ]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class MovieDataCollector:
    def __init__(self, sheet_id=None, worksheet_name=None):
        """Initialize the movie data collector"""
        self.reddit = None

        # TMDB API KEY - INTEGRATED
        self.tmdb_api_key = "c1125eec3c170f3e54968a9c9dd5626c"

        # Google Sheets setup
        self.sheet_id = sheet_id
        self.worksheet_name = worksheet_name
        self.worksheet = None

        # Data storage
        self.movie_data = []
        self.reddit_data = []

        # Time filter (last 3 months)
        self.three_months_ago = datetime.now() - timedelta(days=90)

        # Initialize APIs
        self._setup_apis()
        self._setup_google_sheets()

    def _setup_google_sheets(self):
        """Setup Google Sheets authentication and access"""
        if self.sheet_id and self.worksheet_name:
            try:
                auth.authenticate_user()
                creds, _ = default()
                client = gspread.authorize(creds)
                sheet = client.open_by_key(self.sheet_id)
                self.worksheet = sheet.worksheet(self.worksheet_name)
                logger.info(f"Google Sheets initialized successfully - Worksheet: {self.worksheet_name}")
            except Exception as e:
                logger.error(f"Failed to initialize Google Sheets: {e}")

    def _setup_apis(self):
        """Setup API clients"""
        logger.info("TMDB API key integrated successfully")

        # Reddit API setup - CREDENTIALS NOW CONFIGURED
        try:
            self.reddit = praw.Reddit(
                client_id="HA15MkoZrg4xng2Y7umGsA",
                client_secret="yOJTSrreZG8rruSLOEOxi4XkASS7bw",
                user_agent='movie_data_collector_v1.0'
            )
            logger.info("Reddit API initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize Reddit API: {e}")

    def collect_movie_data(self, pages=5):
        """Collect movie data from TMDB (last 3 months and popular)"""
        logger.info("Starting TMDB movie data collection...")

        # Calculate date range for last 3 months
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = self.three_months_ago.strftime('%Y-%m-%d')

        # Collect recent releases
        logger.info("Collecting recent movie releases...")
        for page in range(1, pages + 1):
            try:
                url = f"https://api.themoviedb.org/3/discover/movie"
                params = {
                    'api_key': self.tmdb_api_key,
                    'page': page,
                    'primary_release_date.gte': start_date,
                    'primary_release_date.lte': end_date,
                    'sort_by': 'popularity.desc',
                    'language': 'en-US'
                }

                response = requests.get(url, params=params)
                if response.status_code == 200:
                    data = response.json()
                    self._process_movies(data['results'])
                    logger.info(f"Collected recent movies - Page {page}")
                else:
                    logger.error(f"Failed to get recent movies page {page}: {response.status_code}")

                time.sleep(1)  # Rate limiting

            except Exception as e:
                logger.error(f"Error collecting recent movies page {page}: {e}")

        # Collect popular movies (regardless of release date)
        logger.info("Collecting popular movies...")
        for page in range(1, pages + 1):
            try:
                url = f"https://api.themoviedb.org/3/movie/popular"
                params = {
                    'api_key': self.tmdb_api_key,
                    'page': page,
                    'language': 'en-US'
                }

                response = requests.get(url, params=params)
                if response.status_code == 200:
                    data = response.json()
                    self._process_movies(data['results'], is_popular=True)
                    logger.info(f"Collected popular movies - Page {page}")
                else:
                    logger.error(f"Failed to get popular movies page {page}: {response.status_code}")

                time.sleep(1)

            except Exception as e:
                logger.error(f"Error collecting popular movies page {page}: {e}")

        # Collect top rated movies
        logger.info("Collecting top rated movies...")
        for page in range(1, min(pages, 3) + 1):
            try:
                url = f"https://api.themoviedb.org/3/movie/top_rated"
                params = {
                    'api_key': self.tmdb_api_key,
                    'page': page,
                    'language': 'en-US'
                }

                response = requests.get(url, params=params)
                if response.status_code == 200:
                    data = response.json()
                    self._process_movies(data['results'], is_top_rated=True)
                    logger.info(f"Collected top rated movies - Page {page}")
                else:
                    logger.error(f"Failed to get top rated movies page {page}: {response.status_code}")

                time.sleep(1)

            except Exception as e:
                logger.error(f"Error collecting top rated movies page {page}: {e}")

        logger.info(f"Total movies collected: {len(self.movie_data)}")

    def _process_movies(self, movies, is_popular=False, is_top_rated=False):
        """Process movie data and get detailed information"""
        for movie in movies:
            try:
                # Check if movie already exists
                if any(m['tmdb_id'] == movie['id'] for m in self.movie_data):
                    continue

                # Get detailed movie info
                detail_url = f"https://api.themoviedb.org/3/movie/{movie['id']}"
                detail_params = {
                    'api_key': self.tmdb_api_key,
                    'append_to_response': 'credits,keywords,videos,reviews,similar'
                }
                detail_response = requests.get(detail_url, params=detail_params)

                if detail_response.status_code == 200:
                    movie_details = detail_response.json()

                    # Get cast and crew
                    cast = []
                    crew = []
                    directors = []

                    if 'credits' in movie_details:
                        cast = [actor['name'] for actor in movie_details['credits'].get('cast', [])[:5]]
                        crew_data = movie_details['credits'].get('crew', [])
                        directors = [person['name'] for person in crew_data if person['job'] == 'Director']
                        crew = [person['name'] for person in crew_data if person['job'] in ['Producer', 'Writer', 'Screenplay']][:3]

                    # Get keywords
                    keywords = []
                    if 'keywords' in movie_details:
                        keywords = [kw['name'] for kw in movie_details['keywords'].get('keywords', [])[:5]]

                    # Get trailer
                    trailer_url = ''
                    if 'videos' in movie_details and movie_details['videos'].get('results'):
                        trailers = [v for v in movie_details['videos']['results'] if v['type'] == 'Trailer']
                        if trailers:
                            trailer_url = f"https://www.youtube.com/watch?v={trailers[0]['key']}"

                    movie_record = {
                        'tmdb_id': movie['id'],
                        'title': movie['title'],
                        'original_title': movie['original_title'],
                        'tagline': movie_details.get('tagline', ''),
                        'overview': (movie['overview'] or '')[:500],
                        'genres': ', '.join([genre['name'] for genre in movie_details.get('genres', [])]),
                        'vote_average': movie['vote_average'],
                        'vote_count': movie['vote_count'],
                        'popularity': movie['popularity'],
                        'release_date': movie['release_date'],
                        'runtime': movie_details.get('runtime'),
                        'budget': movie_details.get('budget'),
                        'revenue': movie_details.get('revenue'),
                        'original_language': movie['original_language'],
                        'spoken_languages': ', '.join([lang['english_name'] for lang in movie_details.get('spoken_languages', [])]),
                        'production_companies': ', '.join([comp['name'] for comp in movie_details.get('production_companies', [])[:3]]),
                        'production_countries': ', '.join([country['name'] for country in movie_details.get('production_countries', [])]),
                        'directors': ', '.join(directors),
                        'cast': ', '.join(cast),
                        'crew': ', '.join(crew),
                        'keywords': ', '.join(keywords),
                        'adult': movie['adult'],
                        'backdrop_path': movie['backdrop_path'],
                        'poster_path': movie['poster_path'],
                        'homepage': movie_details.get('homepage', ''),
                        'imdb_id': movie_details.get('imdb_id', ''),
                        'trailer_url': trailer_url,
                        'tmdb_url': f"https://www.themoviedb.org/movie/{movie['id']}",
                        'is_recent': not (is_popular or is_top_rated),
                        'is_popular': is_popular,
                        'is_top_rated': is_top_rated,
                        'platform': 'movie',
                        'collected_date': datetime.now().isoformat()
                    }
                    self.movie_data.append(movie_record)

                time.sleep(0.25)  # Rate limiting for detailed requests

            except Exception as e:
                logger.error(f"Error processing movie {movie.get('title', 'Unknown')}: {e}")

    def collect_reddit_data(self, subreddits=None, limit=200):
        """Collect movie-related posts from Reddit (last 3 months)"""
        logger.info("Starting Reddit movie data collection...")

        if not self.reddit:
            logger.warning("Reddit API not available - skipping Reddit data collection")
            return

        if not subreddits:
            subreddits = ['movies', 'moviesuggestions', 'truefilm', 'flicks',
                         'horror', 'criterion', 'boxoffice', 'marvelstudios',
                         'DC_Cinematic', 'MovieDetails']

        three_months_timestamp = self.three_months_ago.timestamp()

        # Movie-related keywords for better filtering
        movie_keywords = ['movie', 'film', 'cinema', 'director', 'actor', 'actress',
                         'blockbuster', 'indie', 'thriller', 'comedy', 'drama', 'action',
                         'horror', 'sci-fi', 'fantasy', 'documentary', 'box office',
                         'oscar', 'golden globe', 'netflix', 'disney', 'marvel', 'dc']

        for subreddit_name in subreddits:
            try:
                subreddit = self.reddit.subreddit(subreddit_name)
                posts_collected = 0

                for submission in subreddit.hot(limit=limit * 2):
                    if submission.created_utc >= three_months_timestamp:
                        # Check for movie-related content
                        full_text = f"{submission.title} {submission.selftext}".lower()
                        has_movie_content = any(keyword in full_text for keyword in movie_keywords)

                        # For general movie subreddits, accept all posts
                        # For other subreddits, filter by keywords
                        if subreddit_name in ['movies', 'moviesuggestions', 'truefilm', 'flicks'] or has_movie_content:
                            reddit_record = {
                                'reddit_id': submission.id,
                                'subreddit': subreddit_name,
                                'title': submission.title,
                                'author': str(submission.author) if submission.author else '[deleted]',
                                'content': (submission.selftext or '')[:500],
                                'score': submission.score,
                                'upvote_ratio': submission.upvote_ratio,
                                'num_comments': submission.num_comments,
                                'created_utc': datetime.utcfromtimestamp(submission.created_utc).isoformat(),
                                'url': f"https://reddit.com{submission.permalink}",
                                'flair': submission.link_flair_text,
                                'is_nsfw': submission.over_18,
                                'is_spoiler': submission.spoiler,
                                'platform': 'reddit_movie',
                                'collected_date': datetime.now().isoformat()
                            }
                            self.reddit_data.append(reddit_record)
                            posts_collected += 1

                            if posts_collected >= limit:
                                break

                logger.info(f"Collected {posts_collected} posts from r/{subreddit_name}")
                time.sleep(2)  # Rate limiting

            except Exception as e:
                logger.error(f"Error collecting Reddit data from r/{subreddit_name}: {e}")

    def save_data_to_csv(self, filename_prefix="movie"):
        """Save collected data to CSV files with consistent 'movie' naming"""
        # Save TMDB/movie data as "movie_tmdb.csv"
        if self.movie_data:
            movie_df = pd.DataFrame(self.movie_data)
            movie_filename = f"{filename_prefix}_tmdb.csv"
            movie_df.to_csv(movie_filename, index=False)
            logger.info(f"Saved {len(self.movie_data)} movie records to {movie_filename}")

        # Save Reddit data as "movie_reddit.csv"
        if self.reddit_data:
            reddit_df = pd.DataFrame(self.reddit_data)
            reddit_filename = f"{filename_prefix}_reddit.csv"
            reddit_df.to_csv(reddit_filename, index=False)
            logger.info(f"Saved {len(self.reddit_data)} Reddit records to {reddit_filename}")

    def upload_to_google_sheets(self):
        """Upload all collected data to Google Sheets"""
        if not self.worksheet:
            logger.error("Google Sheets not initialized")
            return

        try:
            all_data = []

            # Create comprehensive headers for all data types
            headers = [
                'Data_Type', 'ID', 'Title', 'Original_Title', 'Tagline', 'Overview',
                'Genres', 'Vote_Average', 'Vote_Count', 'Popularity', 'Release_Date',
                'Runtime', 'Budget', 'Revenue', 'Original_Language', 'Spoken_Languages',
                'Production_Companies', 'Production_Countries', 'Directors', 'Cast',
                'Crew', 'Keywords', 'Adult', 'Backdrop_Path', 'Poster_Path',
                'Homepage', 'IMDB_ID', 'Trailer_URL', 'TMDB_URL', 'Is_Recent',
                'Is_Popular', 'Is_Top_Rated', 'Platform', 'Collected_Date',
                # Reddit specific fields
                'Subreddit', 'Author', 'Content', 'Score', 'Upvote_Ratio',
                'Num_Comments', 'Created_UTC', 'Reddit_URL', 'Flair', 'Is_NSFW', 'Is_Spoiler'
            ]

            all_data.append(headers)

            # Add movie data
            for movie in self.movie_data:
                row = [
                    'MOVIE',
                    movie.get('tmdb_id', ''),
                    movie.get('title', ''),
                    movie.get('original_title', ''),
                    movie.get('tagline', ''),
                    movie.get('overview', ''),
                    movie.get('genres', ''),
                    movie.get('vote_average', ''),
                    movie.get('vote_count', ''),
                    movie.get('popularity', ''),
                    movie.get('release_date', ''),
                    movie.get('runtime', ''),
                    movie.get('budget', ''),
                    movie.get('revenue', ''),
                    movie.get('original_language', ''),
                    movie.get('spoken_languages', ''),
                    movie.get('production_companies', ''),
                    movie.get('production_countries', ''),
                    movie.get('directors', ''),
                    movie.get('cast', ''),
                    movie.get('crew', ''),
                    movie.get('keywords', ''),
                    movie.get('adult', ''),
                    movie.get('backdrop_path', ''),
                    movie.get('poster_path', ''),
                    movie.get('homepage', ''),
                    movie.get('imdb_id', ''),
                    movie.get('trailer_url', ''),
                    movie.get('tmdb_url', ''),
                    movie.get('is_recent', ''),
                    movie.get('is_popular', ''),
                    movie.get('is_top_rated', ''),
                    movie.get('platform', ''),
                    movie.get('collected_date', ''),
                    # Empty Reddit fields
                    '', '', '', '', '', '', '', '', '', '', ''
                ]
                all_data.append(row)

            # Add Reddit data
            for reddit in self.reddit_data:
                row = [
                    'REDDIT',
                    reddit.get('reddit_id', ''),
                    reddit.get('title', ''),
                    '', '', # Empty original title, tagline
                    reddit.get('content', ''),
                    '', # Empty genres
                    reddit.get('score', ''),
                    '', # Empty vote_count
                    '', # Empty popularity
                    '', # Empty release_date
                    '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', # Empty movie fields
                    reddit.get('platform', ''),
                    reddit.get('collected_date', ''),
                    # Reddit specific fields
                    reddit.get('subreddit', ''),
                    reddit.get('author', ''),
                    reddit.get('content', ''),
                    reddit.get('score', ''),
                    reddit.get('upvote_ratio', ''),
                    reddit.get('num_comments', ''),
                    reddit.get('created_utc', ''),
                    reddit.get('url', ''),
                    reddit.get('flair', ''),
                    reddit.get('is_nsfw', ''),
                    reddit.get('is_spoiler', '')
                ]
                all_data.append(row)

            # Clear and update worksheet
            self.worksheet.clear()
            self.worksheet.update(all_data)

            logger.info(f"Uploaded {len(all_data)-1} records to Google Sheets")
            print(f"Your Google Sheet: https://docs.google.com/spreadsheets/d/{self.sheet_id}/edit")

        except Exception as e:
            logger.error(f"Error uploading to Google Sheets: {e}")

    def get_summary_stats(self):
        """Print summary statistics"""
        print("\n=== MOVIE DATA COLLECTION SUMMARY ===")
        print(f"TMDB movie records: {len(self.movie_data)}")
        print(f"Reddit records: {len(self.reddit_data)}")
        print(f"Total records: {len(self.movie_data) + len(self.reddit_data)}")

        if self.movie_data:
            # Show some stats
            genres = []
            for movie in self.movie_data:
                if movie.get('genres'):
                    genres.extend([g.strip() for g in movie['genres'].split(',')])

            top_genres = Counter(genres).most_common(5)
            print(f"Top 5 genres: {top_genres}")

            # Average rating
            ratings = [movie['vote_average'] for movie in self.movie_data if movie.get('vote_average')]
            if ratings:
                avg_rating = sum(ratings) / len(ratings)
                print(f"Average movie rating: {avg_rating:.2f}")

        print(f"Data collection period: Last 3 months from {datetime.now().strftime('%Y-%m-%d')}")

def main():
    """Main function for movie data collection"""
    print("Starting Movie Data Collection for Database_overall...")

    # Your Google Sheets configuration
    SHEET_ID = '1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk'
    WORKSHEET_NAME = 'Movies'

    collector = MovieDataCollector(sheet_id=SHEET_ID, worksheet_name=WORKSHEET_NAME)

    # Collect data
    collector.collect_movie_data(pages=3)
    collector.collect_reddit_data(limit=100)

    # Save and upload
    collector.save_data_to_csv()
    collector.upload_to_google_sheets()
    collector.get_summary_stats()

    print("Movie data collection complete!")
    print(f"Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit")

if __name__ == "__main__":
    print("MOVIE DATA COLLECTOR - READY TO RUN!")
    print("="*60)
    print("✅ TMDB API Key: CONFIGURED")
    print("✅ Reddit API Credentials: CONFIGURED")
    print()
    print("REQUIRED PACKAGES:")
    print("   pip install praw pandas gspread google-auth google-colab requests")
    print()
    print("GOOGLE SHEETS ACCESS:")
    print("   - Make sure you have access to the Google Sheet")
    print("   - Run in Google Colab for easy authentication")
    print()
    print("TO RUN:")
    print("   python movie_collector.py")
    print("   # OR uncomment main() call below")
    print("="*60)

    # Uncomment the next line to run the script
    main()

MOVIE DATA COLLECTOR - READY TO RUN!
✅ TMDB API Key: CONFIGURED
✅ Reddit API Credentials: CONFIGURED

REQUIRED PACKAGES:
   pip install praw pandas gspread google-auth google-colab requests

GOOGLE SHEETS ACCESS:
   - Make sure you have access to the Google Sheet
   - Run in Google Colab for easy authentication

TO RUN:
   python movie_collector.py
   # OR uncomment main() call below
Starting Movie Data Collection for Database_overall...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

/tmp/ipython-input-3222579472.py:270: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'created_utc': datetime.utcfromtimestamp(submission.created_utc).isoformat(),
It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments f

Your Google Sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit

=== MOVIE DATA COLLECTION SUMMARY ===
TMDB movie records: 148
Reddit records: 920
Total records: 1068
Top 5 genres: [('Drama', 59), ('Action', 46), ('Thriller', 39), ('Comedy', 35), ('Romance', 29)]
Average movie rating: 7.46
Data collection period: Last 3 months from 2025-09-16
Movie data collection complete!
Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit


### Video Games

In [ ]:
### Video Games
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class GameDataCollector:
    def __init__(self, sheet_id=None, worksheet_name=None):
        """Initialize the video game data collector"""
        self.reddit = None
        self.twitch_client_id = None
        self.twitch_client_secret = None
        self.twitch_access_token = None

        # Google Sheets setup
        self.sheet_id = sheet_id
        self.worksheet_name = worksheet_name
        self.worksheet = None

        # Data storage
        self.game_data = []
        self.reddit_data = []

        # Time filter (last 3 months)
        self.three_months_ago = datetime.now() - timedelta(days=90)

        # Initialize APIs
        self._setup_apis()
        self._setup_google_sheets()

    def _setup_google_sheets(self):
        """Setup Google Sheets authentication and access"""
        if self.sheet_id and self.worksheet_name:
            try:
                auth.authenticate_user()
                creds, _ = default()
                client = gspread.authorize(creds)
                sheet = client.open_by_key(self.sheet_id)
                self.worksheet = sheet.worksheet(self.worksheet_name)
                logger.info(f"Google Sheets initialized successfully - Worksheet: {self.worksheet_name}")
            except Exception as e:
                logger.error(f"Failed to initialize Google Sheets: {e}")

    def _setup_apis(self):
        """Setup API clients"""
        # Twitch API setup - CREDENTIALS NOW CONFIGURED
        self.twitch_client_id = "b45lzn61sha4sn0dwfzpl2e9supg7w"
        self.twitch_client_secret = "iv0ga8yqvcdrdxk0p1jkq7eobe5ds5"

        logger.info("Twitch credentials configured successfully")

        # Reddit API setup - CREDENTIALS INTEGRATED
        try:
            self.reddit = praw.Reddit(
                client_id="HA15MkoZrg4xng2Y7umGsA",
                client_secret="yOJTSrreZG8rruSLOEOxi4XkASS7bw",
                user_agent='game_data_collector_v1.0'
            )
            logger.info("Reddit API initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize Reddit API: {e}")

    def _get_twitch_access_token(self):
        """Get Twitch access token using client credentials"""
        try:
            url = "https://id.twitch.tv/oauth2/token"
            params = {
                'client_id': self.twitch_client_id,
                'client_secret': self.twitch_client_secret,
                'grant_type': 'client_credentials'
            }
            response = requests.post(url, params=params)

            if response.status_code == 200:
                token_data = response.json()
                self.twitch_access_token = token_data['access_token']
                logger.info("Twitch access token obtained successfully")
                return self.twitch_access_token
            else:
                logger.error(f"Failed to get Twitch access token: {response.status_code}")
                return None
        except Exception as e:
            logger.error(f"Error getting Twitch access token: {e}")
            return None

    def collect_twitch_data(self, limit=100):
        """Collect game data from Twitch API"""
        logger.info("Starting Twitch game data collection...")

        if not self.twitch_access_token:
            self.twitch_access_token = self._get_twitch_access_token()

        if not self.twitch_access_token:
            logger.error("Twitch access token not available - skipping Twitch data collection")
            return

        headers = {
            'Client-ID': self.twitch_client_id,
            'Authorization': f'Bearer {self.twitch_access_token}'
        }

        try:
            # Get top games
            url = "https://api.twitch.tv/helix/games/top"
            params = {'first': min(limit, 100)}  # Twitch API limit is 100 per request

            all_games = []
            cursor = None

            # Collect multiple pages if needed
            pages_needed = (limit + 99) // 100  # Round up division
            for page in range(pages_needed):
                if cursor:
                    params['after'] = cursor

                response = requests.get(url, headers=headers, params=params)

                if response.status_code == 200:
                    games_data = response.json()
                    all_games.extend(games_data['data'])

                    # Get cursor for next page
                    cursor = games_data.get('pagination', {}).get('cursor')
                    if not cursor or len(all_games) >= limit:
                        break

                    time.sleep(0.5)  # Rate limiting
                else:
                    logger.error(f"Failed to get Twitch games page {page + 1}: {response.status_code}")
                    break

            # Process each game to get detailed data
            for i, game in enumerate(all_games[:limit]):
                try:
                    game_id = game['id']

                    # Get streams for this game to get more data
                    streams_url = "https://api.twitch.tv/helix/streams"
                    streams_params = {'game_id': game_id, 'first': 20}
                    streams_response = requests.get(streams_url, headers=headers, params=streams_params)

                    total_viewers = 0
                    stream_count = 0
                    top_streamers = []
                    languages = set()

                    if streams_response.status_code == 200:
                        streams_data = streams_response.json()
                        for stream in streams_data['data']:
                            total_viewers += stream['viewer_count']
                            stream_count += 1
                            top_streamers.append({
                                'user_name': stream['user_name'],
                                'viewer_count': stream['viewer_count'],
                                'title': stream['title'][:100]  # Limit title length
                            })
                            languages.add(stream['language'])

                    # Sort streamers by viewer count
                    top_streamers = sorted(top_streamers, key=lambda x: x['viewer_count'], reverse=True)[:5]

                    # Get additional game info if available through IGDB (if game has IGDB ID)
                    igdb_id = game.get('igdb_id')

                    game_record = {
                        'twitch_game_id': game_id,
                        'name': game['name'],
                        'box_art_url': game['box_art_url'],
                        'igdb_id': igdb_id,
                        'current_viewers': total_viewers,
                        'stream_count': stream_count,
                        'avg_viewers_per_stream': round(total_viewers / stream_count, 2) if stream_count > 0 else 0,
                        'top_streamer': top_streamers[0]['user_name'] if top_streamers else '',
                        'top_streamer_viewers': top_streamers[0]['viewer_count'] if top_streamers else 0,
                        'top_streamers': ', '.join([f"{s['user_name']} ({s['viewer_count']})" for s in top_streamers[:3]]),
                        'streaming_languages': ', '.join(list(languages)[:5]),
                        'twitch_rank': i + 1,  # Rank based on Twitch popularity
                        'twitch_url': f"https://www.twitch.tv/directory/game/{game['name'].replace(' ', '%20')}",
                        'platform': 'game',
                        'collected_date': datetime.now().isoformat()
                    }
                    self.game_data.append(game_record)

                    time.sleep(0.5)  # Rate limiting between detailed requests

                except Exception as e:
                    logger.error(f"Error processing game {game.get('name', 'Unknown')}: {e}")

                # Progress update
                if (i + 1) % 20 == 0:
                    logger.info(f"Processed {i + 1}/{len(all_games[:limit])} games")

            logger.info(f"Collected {len(self.game_data)} games from Twitch")

        except Exception as e:
            logger.error(f"Error collecting Twitch data: {e}")

    def collect_reddit_data(self, subreddits=None, limit=200):
        """Collect gaming-related posts from Reddit (last 3 months)"""
        logger.info("Starting Reddit gaming data collection...")

        if not self.reddit:
            logger.error("Reddit API not available")
            return

        if not subreddits:
            subreddits = ['gaming', 'Games', 'pcgaming', 'NintendoSwitch', 'PS5', 'xbox',
                         'Steam', 'GameDeals', 'patientgamers', 'IndieGaming', 'JRPG',
                         'rpg_gamers', 'gamedev', 'speedrun', 'Twitch', 'livestreamfail']

        three_months_timestamp = self.three_months_ago.timestamp()

        # Gaming-related keywords for better filtering
        gaming_keywords = ['game', 'gaming', 'gamer', 'play', 'steam', 'xbox', 'playstation',
                          'nintendo', 'pc gaming', 'console', 'indie', 'aaa', 'multiplayer',
                          'single player', 'rpg', 'fps', 'mmo', 'battle royale', 'sandbox',
                          'platformer', 'puzzle', 'strategy', 'simulation', 'racing',
                          'twitch', 'streamer', 'gameplay', 'speedrun', 'esports']

        for subreddit_name in subreddits:
            try:
                subreddit = self.reddit.subreddit(subreddit_name)
                posts_collected = 0

                for submission in subreddit.hot(limit=limit * 2):
                    if submission.created_utc >= three_months_timestamp:
                        # Check for gaming-related content
                        full_text = f"{submission.title} {submission.selftext}".lower()
                        has_gaming_content = any(keyword in full_text for keyword in gaming_keywords)

                        # For gaming subreddits, accept all posts
                        # For other subreddits, filter by keywords
                        gaming_subreddits = ['gaming', 'Games', 'pcgaming', 'Steam', 'gamedev', 'IndieGaming']
                        if subreddit_name in gaming_subreddits or has_gaming_content:
                            # Try to detect specific games mentioned
                            detected_games = self._detect_games_in_text(full_text)

                            reddit_record = {
                                'reddit_id': submission.id,
                                'subreddit': subreddit_name,
                                'title': submission.title,
                                'author': str(submission.author) if submission.author else '[deleted]',
                                'content': (submission.selftext or '')[:500],
                                'score': submission.score,
                                'upvote_ratio': submission.upvote_ratio,
                                'num_comments': submission.num_comments,
                                'created_utc': datetime.utcfromtimestamp(submission.created_utc).isoformat(),
                                'url': f"https://reddit.com{submission.permalink}",
                                'flair': submission.link_flair_text,
                                'is_nsfw': submission.over_18,
                                'is_spoiler': submission.spoiler,
                                'domain': submission.domain,
                                'is_video': submission.is_video,
                                'detected_games': ', '.join(detected_games) if detected_games else '',
                                'game_count': len(detected_games),
                                'platform': 'reddit_gaming',
                                'collected_date': datetime.now().isoformat()
                            }
                            self.reddit_data.append(reddit_record)
                            posts_collected += 1

                            if posts_collected >= limit:
                                break

                logger.info(f"Collected {posts_collected} posts from r/{subreddit_name}")
                time.sleep(2)  # Rate limiting

            except Exception as e:
                logger.error(f"Error collecting Reddit data from r/{subreddit_name}: {e}")

    def _detect_games_in_text(self, text):
        """Simple game detection in text"""
        # Popular game titles (you can expand this list)
        game_titles = [
            'minecraft', 'fortnite', 'call of duty', 'grand theft auto', 'gta',
            'the witcher', 'cyberpunk 2077', 'red dead redemption', 'fifa', 'nba 2k',
            'league of legends', 'dota 2', 'counter-strike', 'valorant', 'overwatch',
            'apex legends', 'destiny 2', 'world of warcraft', 'wow', 'final fantasy',
            'zelda', 'mario', 'pokemon', 'smash bros', 'animal crossing',
            'among us', 'fall guys', 'rocket league', 'rainbow six siege',
            'pubg', 'battlefield', 'assassins creed', 'dark souls', 'elden ring',
            'god of war', 'spider-man', 'horizon', 'last of us', 'uncharted',
            'halo', 'gears of war', 'forza', 'sea of thieves', 'minecraft dungeons'
        ]

        detected = []
        for game in game_titles:
            if game in text:
                detected.append(game.title())

        return list(set(detected))  # Remove duplicates

    def save_data_to_csv(self, filename_prefix="video_game"):
        """Save collected data to CSV files with consistent 'video game' naming"""
        # Save Twitch/game data as "video_game_twitch.csv"
        if self.game_data:
            game_df = pd.DataFrame(self.game_data)
            game_filename = f"{filename_prefix}_twitch.csv"
            game_df.to_csv(game_filename, index=False)
            logger.info(f"Saved {len(self.game_data)} game records to {game_filename}")

        # Save Reddit data as "video_game_reddit.csv"
        if self.reddit_data:
            reddit_df = pd.DataFrame(self.reddit_data)
            reddit_filename = f"{filename_prefix}_reddit.csv"
            reddit_df.to_csv(reddit_filename, index=False)
            logger.info(f"Saved {len(self.reddit_data)} Reddit records to {reddit_filename}")

    def upload_to_google_sheets(self):
        """Upload all collected data to Google Sheets"""
        if not self.worksheet:
            logger.error("Google Sheets not initialized")
            return

        try:
            all_data = []

            # Create comprehensive headers for all data types
            headers = [
                'Data_Type', 'ID', 'Name', 'Box_Art_URL', 'IGDB_ID',
                'Current_Viewers', 'Stream_Count', 'Avg_Viewers_Per_Stream',
                'Top_Streamer', 'Top_Streamer_Viewers', 'Top_Streamers',
                'Streaming_Languages', 'Twitch_Rank', 'Twitch_URL', 'Platform', 'Collected_Date',
                # Reddit specific fields
                'Subreddit', 'Title', 'Author', 'Content', 'Score', 'Upvote_Ratio',
                'Num_Comments', 'Created_UTC', 'Reddit_URL', 'Flair', 'Is_NSFW',
                'Is_Spoiler', 'Domain', 'Is_Video', 'Detected_Games', 'Game_Count'
            ]

            all_data.append(headers)

            # Add game data
            for game in self.game_data:
                row = [
                    'GAME',
                    game.get('twitch_game_id', ''),
                    game.get('name', ''),
                    game.get('box_art_url', ''),
                    game.get('igdb_id', ''),
                    game.get('current_viewers', ''),
                    game.get('stream_count', ''),
                    game.get('avg_viewers_per_stream', ''),
                    game.get('top_streamer', ''),
                    game.get('top_streamer_viewers', ''),
                    game.get('top_streamers', ''),
                    game.get('streaming_languages', ''),
                    game.get('twitch_rank', ''),
                    game.get('twitch_url', ''),
                    game.get('platform', ''),
                    game.get('collected_date', ''),
                    # Empty Reddit fields
                    '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''
                ]
                all_data.append(row)

            # Add Reddit data
            for reddit in self.reddit_data:
                row = [
                    'REDDIT',
                    reddit.get('reddit_id', ''),
                    reddit.get('title', ''),
                    '', '', # Empty box art, IGDB ID
                    '', '', '', '', '', '', '', '', '', # Empty game-specific fields
                    reddit.get('platform', ''),
                    reddit.get('collected_date', ''),
                    # Reddit specific fields
                    reddit.get('subreddit', ''),
                    reddit.get('title', ''),
                    reddit.get('author', ''),
                    reddit.get('content', ''),
                    reddit.get('score', ''),
                    reddit.get('upvote_ratio', ''),
                    reddit.get('num_comments', ''),
                    reddit.get('created_utc', ''),
                    reddit.get('url', ''),
                    reddit.get('flair', ''),
                    reddit.get('is_nsfw', ''),
                    reddit.get('is_spoiler', ''),
                    reddit.get('domain', ''),
                    reddit.get('is_video', ''),
                    reddit.get('detected_games', ''),
                    reddit.get('game_count', '')
                ]
                all_data.append(row)

            # Clear and update worksheet
            self.worksheet.clear()
            self.worksheet.update(all_data)

            logger.info(f"Uploaded {len(all_data)-1} records to Google Sheets")
            print(f"Your Google Sheet: https://docs.google.com/spreadsheets/d/{self.sheet_id}/edit")

        except Exception as e:
            logger.error(f"Error uploading to Google Sheets: {e}")

    def get_summary_stats(self):
        """Print summary statistics"""
        print("\n=== GAME DATA COLLECTION SUMMARY ===")
        print(f"Twitch game records: {len(self.game_data)}")
        print(f"Reddit records: {len(self.reddit_data)}")
        print(f"Total records: {len(self.game_data) + len(self.reddit_data)}")

        if self.game_data:
            # Show some stats
            total_viewers = sum(game.get('current_viewers', 0) for game in self.game_data)
            total_streams = sum(game.get('stream_count', 0) for game in self.game_data)

            print(f"Total viewers across all games: {total_viewers:,}")
            print(f"Total active streams: {total_streams:,}")

            # Top games by viewer count
            top_games = sorted(self.game_data, key=lambda x: x.get('current_viewers', 0), reverse=True)[:5]
            print("Top 5 games by current viewers:")
            for i, game in enumerate(top_games):
                print(f"  {i+1}. {game.get('name')} - {game.get('current_viewers', 0):,} viewers")

        if self.reddit_data:
            # Reddit stats
            gaming_mentions = {}
            for post in self.reddit_data:
                games = post.get('detected_games', '')
                if games:
                    for game in games.split(', '):
                        gaming_mentions[game] = gaming_mentions.get(game, 0) + 1

            if gaming_mentions:
                top_mentioned = Counter(gaming_mentions).most_common(5)
                print(f"Most mentioned games on Reddit: {top_mentioned}")

        print(f"Data collection period: Last 3 months from {datetime.now().strftime('%Y-%m-%d')}")

def main():
    """Main function for game data collection"""
    print("Starting Video Game Data Collection for Database_overall...")

    # Your Google Sheets configuration
    SHEET_ID = '1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk'
    WORKSHEET_NAME = 'Video_Games'

    collector = GameDataCollector(sheet_id=SHEET_ID, worksheet_name=WORKSHEET_NAME)

    # Collect data
    collector.collect_twitch_data(limit=50)  # Top 50 games
    collector.collect_reddit_data(limit=100)

    # Save and upload
    collector.save_data_to_csv()
    collector.upload_to_google_sheets()
    collector.get_summary_stats()

    print("Game data collection complete!")
    print(f"Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit")

if __name__ == "__main__":
    print("GAME DATA COLLECTOR - READY TO RUN!")
    print("="*60)
    print("✅ Twitch API Credentials: CONFIGURED")
    print("✅ Reddit API Credentials: CONFIGURED")
    print()
    print("REQUIRED PACKAGES:")
    print("   pip install praw pandas gspread google-auth google-colab requests")
    print()
    print("GOOGLE SHEETS ACCESS:")
    print("   - Make sure you have access to the Google Sheet")
    print("   - Run in Google Colab for easy authentication")
    print()
    print("TO RUN:")
    print("   python game_collector.py")
    print("   # OR uncomment main() call below")
    print("="*60)

    # Uncomment the next line to run the script
    main()

GAME DATA COLLECTOR - READY TO RUN!
✅ Twitch API Credentials: CONFIGURED
✅ Reddit API Credentials: CONFIGURED

REQUIRED PACKAGES:
   pip install praw pandas gspread google-auth google-colab requests

GOOGLE SHEETS ACCESS:
   - Make sure you have access to the Google Sheet
   - Run in Google Colab for easy authentication

TO RUN:
   python game_collector.py
   # OR uncomment main() call below
Starting Video Game Data Collection for Database_overall...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

/tmp/ipython-input-251890252.py:245: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'created_utc': datetime.utcfromtimestamp(submission.created_utc).isoformat(),
It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments fo

Your Google Sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit

=== GAME DATA COLLECTION SUMMARY ===
Twitch game records: 50
Reddit records: 1542
Total records: 1592
Total viewers across all games: 1,598,228
Total active streams: 977
Top 5 games by current viewers:
  1. Just Chatting - 346,701 viewers
  2. EA Sports FC 26 - 156,524 viewers
  3. IRL - 123,710 viewers
  4. League of Legends - 64,136 viewers
  5. Grand Theft Auto V - 63,450 viewers
Most mentioned games on Reddit: [('Mario', 48), ('Final Fantasy', 42), ('Battlefield', 19), ('Gta', 14), ('Pokemon', 14)]
Data collection period: Last 3 months from 2025-09-16
Game data collection complete!
Check your sheet: https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit


## Model

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ContentRecommendationService:
    """
    Enhanced recommendation service with higher confidence scores
    """

    def __init__(self, csv_files: Dict[str, str] = None, google_sheets_url: str = None):
        """Initialize the recommendation service with enhanced scoring"""
        self.content_database = []
        self.content_index = {}
        self.platform_index = {}
        self.theme_index = {}
        self.genre_index = {}

        # Enhanced recommendation weights for higher confidence
        self.weights = {
            'content_based': 0.45,
            'collaborative': 0.20,
            'popularity': 0.20,
            'cross_platform_bonus': 0.15
        }

        # Enhanced scoring parameters
        self.confidence_boost = 0.25
        self.min_confidence = 50
        self.quality_threshold = 0.6

        # Default CSV file mappings
        self.csv_files = csv_files or {
            'anime': 'anime_reddit.csv',
            'movie_reddit': 'movie_reddit.csv',
            'movie_tmdb': 'movie_tmdb.csv',
            'game_reddit': 'video_game_reddit.csv',
            'game_twitch': 'video_game_twitch.csv'
        }

        self.google_sheets_url = google_sheets_url

        self._load_and_process_data()
        self._build_indexes()
        self._simulate_collaborative_data()
        self._build_similarity_cache()

    def _load_google_sheets_data(self) -> pd.DataFrame:
        """Load data from Google Sheets"""
        if not self.google_sheets_url:
            return pd.DataFrame()

        try:
            # Convert Google Sheets URL to CSV export format
            if '/edit' in self.google_sheets_url:
                csv_url = self.google_sheets_url.replace('/edit?usp=sharing', '/export?format=csv')
                csv_url = csv_url.replace('/edit#gid=', '/export?format=csv&gid=')
            else:
                csv_url = self.google_sheets_url

            response = requests.get(csv_url, timeout=10)
            response.raise_for_status()

            df = pd.read_csv(StringIO(response.text))
            logger.info(f"Loaded {len(df)} rows from Google Sheets")
            return df

        except Exception as e:
            logger.error(f"Error loading Google Sheets data: {e}")
            return pd.DataFrame()

    def _load_and_process_data(self):
        """Load and standardize data from all sources"""
        try:
            all_data = []

            # Load from CSV files
            for data_type, file_path in self.csv_files.items():
                if data_type == 'anime':
                    data = self._load_anime_data(file_path)
                elif data_type in ['movie_reddit', 'movie_tmdb']:
                    data = self._load_movie_data(file_path, source=data_type)
                elif data_type in ['game_reddit', 'game_twitch']:
                    data = self._load_game_data(file_path, source=data_type)

                all_data.extend(data)
                logger.info(f"Loaded {len(data)} items from {file_path}")

            # Load from Google Sheets if available
            if self.google_sheets_url:
                sheets_df = self._load_google_sheets_data()
                if not sheets_df.empty:
                    sheets_data = self._process_sheets_data(sheets_df)
                    all_data.extend(sheets_data)
                    logger.info(f"Added {len(sheets_data)} items from Google Sheets")

            self.content_database = all_data
            logger.info(f"Total loaded: {len(self.content_database)} items")

        except Exception as e:
            logger.error(f"Error loading data: {e}")
            self._create_fallback_data()

    def _process_sheets_data(self, df: pd.DataFrame) -> List[Dict]:
        """Process data from Google Sheets with flexible column detection"""
        items = []

        # Detect columns by common patterns
        title_cols = [col for col in df.columns if any(keyword in col.lower()
                     for keyword in ['title', 'name', 'show', 'movie', 'game'])]
        platform_cols = [col for col in df.columns if any(keyword in col.lower()
                        for keyword in ['platform', 'type', 'category', 'source'])]
        genre_cols = [col for col in df.columns if any(keyword in col.lower()
                     for keyword in ['genre', 'genres', 'category', 'tags'])]
        rating_cols = [col for col in df.columns if any(keyword in col.lower()
                      for keyword in ['rating', 'score', 'vote', 'points'])]
        desc_cols = [col for col in df.columns if any(keyword in col.lower()
                    for keyword in ['description', 'summary', 'overview', 'plot', 'synopsis'])]

        for idx, row in df.iterrows():
            try:
                # Extract title
                title = 'Unknown'
                for col in title_cols:
                    if pd.notna(row[col]) and str(row[col]).strip():
                        title = str(row[col]).strip()
                        break

                if title == 'Unknown' or title == 'nan':
                    continue

                # Extract platform/type
                platform = 'unknown'
                for col in platform_cols:
                    if pd.notna(row[col]) and str(row[col]).strip():
                        platform = str(row[col]).lower().strip()
                        break

                # Map platform names
                platform_mapping = {
                    'anime': 'anime',
                    'movie': 'movie',
                    'film': 'movie',
                    'game': 'game',
                    'video game': 'game',
                    'tv': 'tv',
                    'series': 'tv'
                }
                platform = platform_mapping.get(platform, platform)

                # Extract rating/score
                score = 0.0
                for col in rating_cols:
                    if pd.notna(row[col]):
                        try:
                            score = float(row[col])
                            break
                        except (ValueError, TypeError):
                            continue

                # Extract genres
                genres = []
                for col in genre_cols:
                    if pd.notna(row[col]) and str(row[col]).strip():
                        genres = self._parse_list_field(str(row[col]))
                        break

                # Extract description
                description = ''
                for col in desc_cols:
                    if pd.notna(row[col]) and str(row[col]).strip():
                        description = str(row[col]).strip()
                        break

                # Process the item
                themes = self._extract_themes(description, genres)
                moods = self._extract_moods(genres, themes, description)

                # Normalize score to 0-1 range
                if score > 10:
                    popularity_score = min(1.0, max(0.1, score / 100.0))
                else:
                    popularity_score = min(1.0, max(0.1, score / 10.0)) if score > 0 else 0.5

                item = {
                    'content_id': f'sheets_{platform}_{idx}',
                    'title': title,
                    'platform': platform,
                    'genres': genres,
                    'themes': themes,
                    'moods': moods,
                    'popularity_score': popularity_score,
                    'engagement_score': popularity_score * 1.0,
                    'description': description[:300] + '...' if len(description) > 300 else description,
                    'original_score': score,
                    'source': 'google_sheets'
                }

                items.append(item)

            except Exception as e:
                logger.warning(f"Error processing sheets row {idx}: {e}")
                continue

        return items

    def _load_anime_data(self, file_path: str) -> List[Dict]:
        """Load and process anime data"""
        try:
            df = pd.read_csv(file_path)
            anime_items = []

            for idx, row in df.iterrows():
                try:
                    # Try different possible title columns
                    title_cols = ['title', 'name', 'anime_title', 'show_name']
                    title = None
                    for col in title_cols:
                        if col in df.columns and pd.notna(row[col]):
                            title = str(row[col]).strip()
                            if title and title != 'nan':
                                break

                    if not title:
                        continue

                    # Try different score columns
                    score = self._extract_score(row, ['score', 'rating', 'mal_score', 'user_score', 'average_score'])
                    genres = self._extract_genres(row)
                    description = self._extract_description(row, ['synopsis', 'description', 'plot', 'summary', 'overview'])

                    themes = self._extract_themes(description, genres)
                    moods = self._extract_moods(genres, themes, description)
                    popularity_score = min(1.0, max(0.1, score / 10.0)) if score > 0 else 0.5

                    item = {
                        'content_id': f'anime_{idx}',
                        'title': title,
                        'platform': 'anime',
                        'genres': genres,
                        'themes': themes,
                        'moods': moods,
                        'popularity_score': popularity_score,
                        'engagement_score': popularity_score * 0.95,
                        'description': description[:300] + '...' if len(description) > 300 else description,
                        'original_score': score,
                        'source': 'reddit'
                    }

                    anime_items.append(item)

                except Exception as e:
                    continue

            return anime_items

        except FileNotFoundError:
            logger.warning(f"Anime CSV not found: {file_path}")
            return []
        except Exception as e:
            logger.error(f"Error loading anime data: {e}")
            return []

    def _load_game_data(self, file_path: str, source: str = 'reddit') -> List[Dict]:
        """Load and process game data"""
        try:
            df = pd.read_csv(file_path)
            game_items = []

            for idx, row in df.iterrows():
                try:
                    # Try different possible title columns
                    title_cols = ['title', 'name', 'game_title', 'game_name', 'game']
                    title = None
                    for col in title_cols:
                        if col in df.columns and pd.notna(row[col]):
                            title = str(row[col]).strip()
                            if title and title != 'nan':
                                break

                    if not title:
                        continue

                    # Try different score columns based on source
                    if 'twitch' in source:
                        score_cols = ['viewers', 'avg_viewers', 'peak_viewers', 'hours_watched']
                    else:
                        score_cols = ['rating', 'score', 'metacritic_score', 'user_score', 'upvotes']

                    score = self._extract_score(row, score_cols)
                    genres = self._extract_genres(row)
                    description = self._extract_description(row, ['description', 'summary', 'plot', 'overview', 'about'])

                    themes = self._extract_themes(description, genres)
                    moods = self._extract_moods(genres, themes, description)

                    # Handle different score ranges
                    if 'twitch' in source and score > 1000:  # Viewer counts
                        popularity_score = min(1.0, max(0.1, np.log10(score + 1) / 6))
                    elif score > 10:
                        popularity_score = min(1.0, max(0.1, score / 100.0))
                    else:
                        popularity_score = min(1.0, max(0.1, score / 10.0)) if score > 0 else 0.5

                    item = {
                        'content_id': f'game_{source}_{idx}',
                        'title': title,
                        'platform': 'game',
                        'genres': genres,
                        'themes': themes,
                        'moods': moods,
                        'popularity_score': popularity_score,
                        'engagement_score': popularity_score * 1.1,
                        'description': description[:300] + '...' if len(description) > 300 else description,
                        'original_score': score,
                        'source': source
                    }

                    game_items.append(item)

                except Exception as e:
                    continue

            return game_items

        except FileNotFoundError:
            logger.warning(f"Game CSV not found: {file_path}")
            return []
        except Exception as e:
            logger.error(f"Error loading game data: {e}")
            return []

    def _load_movie_data(self, file_path: str, source: str = 'reddit') -> List[Dict]:
        """Load and process movie data"""
        try:
            df = pd.read_csv(file_path)
            movie_items = []

            for idx, row in df.iterrows():
                try:
                    # Try different possible title columns
                    title_cols = ['title', 'name', 'movie_title', 'film_title', 'original_title']
                    title = None
                    for col in title_cols:
                        if col in df.columns and pd.notna(row[col]):
                            title = str(row[col]).strip()
                            if title and title != 'nan':
                                break

                    if not title:
                        continue

                    # Try different score columns based on source
                    if 'tmdb' in source:
                        score_cols = ['vote_average', 'rating', 'popularity']
                    else:
                        score_cols = ['rating', 'score', 'imdb_rating', 'upvotes', 'user_score']

                    score = self._extract_score(row, score_cols)
                    genres = self._extract_genres(row)
                    description = self._extract_description(row, ['overview', 'plot', 'description', 'summary', 'synopsis'])

                    themes = self._extract_themes(description, genres)
                    moods = self._extract_moods(genres, themes, description)
                    popularity_score = min(1.0, max(0.1, score / 10.0)) if score > 0 else 0.5

                    item = {
                        'content_id': f'movie_{source}_{idx}',
                        'title': title,
                        'platform': 'movie',
                        'genres': genres,
                        'themes': themes,
                        'moods': moods,
                        'popularity_score': popularity_score,
                        'engagement_score': popularity_score * 0.98,
                        'description': description[:300] + '...' if len(description) > 300 else description,
                        'original_score': score,
                        'source': source
                    }

                    movie_items.append(item)

                except Exception as e:
                    continue

            return movie_items

        except FileNotFoundError:
            logger.warning(f"Movie CSV not found: {file_path}")
            return []
        except Exception as e:
            logger.error(f"Error loading movie data: {e}")
            return []

    def _extract_score(self, row: pd.Series, score_columns: List[str]) -> float:
        """Extract numerical score from row"""
        for col in score_columns:
            if col in row.index and pd.notna(row[col]):
                try:
                    value = str(row[col]).replace(',', '').strip()
                    return float(value)
                except (ValueError, TypeError):
                    continue
        return 0.0

    def _extract_genres(self, row: pd.Series) -> List[str]:
        """Extract genres from row data"""
        for col in ['genres', 'genre', 'category', 'tags', 'categories']:
            if col in row.index and pd.notna(row[col]):
                return self._parse_list_field(str(row[col]))
        return []

    def _extract_description(self, row: pd.Series, desc_columns: List[str]) -> str:
        """Extract description text from row"""
        for col in desc_columns:
            if col in row.index and pd.notna(row[col]):
                desc = str(row[col]).strip()
                if desc and desc != 'nan' and len(desc) > 10:
                    return desc
        return ""

    def _parse_list_field(self, field_value: str) -> List[str]:
        """Parse list-like string fields"""
        if not field_value or field_value == 'nan':
            return []

        field_value = str(field_value).strip().lower()

        # Handle JSON-like lists
        if field_value.startswith('[') and field_value.endswith(']'):
            try:
                clean_value = field_value.replace("'", '"')
                items = json.loads(clean_value)
                return [str(item).strip() for item in items if str(item).strip()]
            except:
                # Remove brackets and split
                field_value = field_value[1:-1]

        # Handle common delimiters
        for delimiter in [',', '|', ';', '/']:
            if delimiter in field_value:
                items = [item.strip() for item in field_value.split(delimiter)]
                return [item for item in items if item and item != 'nan']

        return [field_value.strip()] if field_value.strip() else []

    def _extract_themes(self, description: str, genres: List[str]) -> List[str]:
        """Extract thematic elements using keyword matching"""
        if not description:
            return []

        desc_lower = description.lower()
        detected_themes = []

        theme_patterns = {
            'survival': ['survival', 'survive', 'post-apocalyptic', 'wasteland', 'zombie', 'disaster'],
            'moral_ambiguity': ['moral', 'ethics', 'choice', 'dilemma', 'right and wrong', 'gray area'],
            'power_struggle': ['power', 'control', 'domination', 'authority', 'politics', 'conspiracy'],
            'friendship': ['friend', 'friendship', 'bond', 'companion', 'brotherhood', 'loyalty'],
            'family': ['family', 'father', 'mother', 'brother', 'sister', 'parent', 'child'],
            'love_romance': ['love', 'romance', 'romantic', 'relationship', 'heart', 'passion'],
            'war_conflict': ['war', 'battle', 'conflict', 'military', 'soldier', 'combat'],
            'revenge': ['revenge', 'vengeance', 'payback', 'retribution', 'vendetta'],
            'justice': ['justice', 'law', 'crime', 'punishment', 'court', 'police'],
            'identity': ['identity', 'who am i', 'self-discovery', 'finding yourself'],
            'sacrifice': ['sacrifice', 'give up', 'loss', 'cost', 'price'],
            'redemption': ['redemption', 'forgiveness', 'second chance', 'atonement'],
            'coming_of_age': ['growing up', 'teenager', 'high school', 'adolescent', 'young adult'],
            'supernatural': ['magic', 'supernatural', 'ghost', 'spirit', 'demon', 'occult'],
            'technology': ['technology', 'ai', 'robot', 'cyber', 'digital', 'hacker'],
            'time_travel': ['time travel', 'past', 'future', 'timeline', 'temporal'],
            'mystery': ['mystery', 'detective', 'investigation', 'clue', 'solve', 'murder']
        }

        for theme, keywords in theme_patterns.items():
            if any(keyword in desc_lower for keyword in keywords):
                detected_themes.append(theme)

        # Map genres to themes
        genre_to_theme = {
            'horror': 'supernatural',
            'thriller': 'mystery',
            'romance': 'love_romance',
            'war': 'war_conflict',
            'military': 'war_conflict',
            'crime': 'justice',
            'detective': 'mystery',
            'sci_fi': 'technology',
            'science fiction': 'technology',
            'fantasy': 'supernatural'
        }

        for genre in genres:
            if genre in genre_to_theme:
                detected_themes.append(genre_to_theme[genre])

        return list(set(detected_themes))[:8]

    def _extract_moods(self, genres: List[str], themes: List[str], description: str) -> List[str]:
        """Extract emotional mood/tone"""
        moods = set()

        genre_moods = {
            'horror': 'dark',
            'thriller': 'intense',
            'action': 'intense',
            'drama': 'emotional',
            'comedy': 'light_hearted',
            'romance': 'romantic',
            'sci_fi': 'philosophical',
            'science fiction': 'philosophical',
            'mystery': 'mysterious',
            'war': 'dark',
            'crime': 'dark'
        }

        for genre in genres:
            if genre in genre_moods:
                moods.add(genre_moods[genre])

        theme_moods = {
            'survival': 'dark',
            'war_conflict': 'dark',
            'moral_ambiguity': 'philosophical',
            'power_struggle': 'intense',
            'love_romance': 'romantic',
            'friendship': 'emotional',
            'family': 'emotional',
            'mystery': 'mysterious'
        }

        for theme in themes:
            if theme in theme_moods:
                moods.add(theme_moods[theme])

        if description:
            desc_lower = description.lower()
            if any(word in desc_lower for word in ['dark', 'death', 'violence', 'brutal']):
                moods.add('dark')
            if any(word in desc_lower for word in ['emotional', 'touching', 'heartfelt']):
                moods.add('emotional')
            if any(word in desc_lower for word in ['funny', 'comedy', 'humorous']):
                moods.add('light_hearted')
            if any(word in desc_lower for word in ['intense', 'thrilling', 'exciting']):
                moods.add('intense')

        return list(moods)

    def _build_indexes(self):
        """Build fast lookup indexes"""
        self.content_index = {}
        self.platform_index = {}
        self.theme_index = {}
        self.genre_index = {}

        for item in self.content_database:
            title_key = item['title'].lower()
            self.content_index[title_key] = item

        for item in self.content_database:
            platform = item['platform']
            if platform not in self.platform_index:
                self.platform_index[platform] = []
            self.platform_index[platform].append(item)

        for item in self.content_database:
            for theme in item['themes']:
                if theme not in self.theme_index:
                    self.theme_index[theme] = []
                self.theme_index[theme].append(item)

        for item in self.content_database:
            for genre in item['genres']:
                if genre not in self.genre_index:
                    self.genre_index[genre] = []
                self.genre_index[genre].append(item)

    def _simulate_collaborative_data(self):
        """Generate simulated user interaction data"""
        import random
        random.seed(42)
        self.user_interactions = {}

        for i in range(100):
            user_items = []
            num_items = random.randint(3, 12)
            available_items = random.sample(self.content_database,
                                          min(num_items * 2, len(self.content_database)))

            weighted_items = []
            for item in available_items:
                weight = int(item['popularity_score'] * 10) + 1
                weighted_items.extend([item['content_id']] * weight)

            if weighted_items:
                user_items = list(set(random.choices(weighted_items, k=num_items)))

            if user_items:
                self.user_interactions[f'user_{i}'] = user_items

    def _create_fallback_data(self):
        """Create minimal fallback data"""
        self.content_database = [
            {
                'content_id': 'fallback_1',
                'title': 'Sample Content',
                'platform': 'unknown',
                'genres': ['action', 'drama'],
                'themes': ['friendship', 'power_struggle'],
                'moods': ['intense', 'emotional'],
                'popularity_score': 0.8,
                'engagement_score': 0.75,
                'description': 'Sample content for testing',
                'source': 'fallback'
            }
        ]
        self._build_indexes()

    def _build_similarity_cache(self):
        """Pre-compute similarity scores"""
        all_genres = set()
        for item in self.content_database:
            all_genres.update(item['genres'])

        self.genre_weights = {}
        for genre in all_genres:
            genre_count = sum(1 for item in self.content_database if genre in item['genres'])
            self.genre_weights[genre] = max(0.5, min(2.0, 50 / max(genre_count, 1)))

    def find_content(self, title: str) -> Optional[Dict]:
        """Find content by title"""
        title_key = title.lower().strip()

        if title_key in self.content_index:
            return self.content_index[title_key]

        # Fuzzy matching
        for indexed_title, content in self.content_index.items():
            if title_key in indexed_title or indexed_title in title_key:
                return content

        return None

    def _calculate_enhanced_content_similarity(self, item1: Dict, item2: Dict) -> float:
        """Enhanced content similarity with weighted scoring"""
        genres1, genres2 = set(item1['genres']), set(item2['genres'])
        if genres1 and genres2:
            common_genres = genres1 & genres2
            genre_score = 0
            for genre in common_genres:
                genre_score += self.genre_weights.get(genre, 1.0)
            genre_sim = min(1.0, genre_score / len(genres1 | genres2))
        else:
            genre_sim = 0

        themes1, themes2 = set(item1['themes']), set(item2['themes'])
        if themes1 and themes2:
            theme_overlap = len(themes1 & themes2)
            theme_total = len(themes1 | themes2)
            theme_sim = (theme_overlap / theme_total) ** 0.8 if theme_total > 0 else 0
            if theme_overlap >= 2:
                theme_sim = min(1.0, theme_sim * 1.3)
        else:
            theme_sim = 0

        moods1, moods2 = set(item1['moods']), set(item2['moods'])
        if moods1 and moods2:
            mood_overlap = len(moods1 & moods2)
            mood_sim = mood_overlap / len(moods1 | moods2) if moods1 | moods2 else 0
            if mood_overlap >= 2:
                mood_sim = min(1.0, mood_sim * 1.2)
        else:
            mood_sim = 0

        pop_diff = abs(item1['popularity_score'] - item2['popularity_score'])
        pop_sim = max(0, 1 - pop_diff * 2)

        content_sim = (
            theme_sim * 0.45 +
            genre_sim * 0.35 +
            mood_sim * 0.15 +
            pop_sim * 0.05
        )

        return min(1.0, content_sim)

    def _calculate_enhanced_collaborative_score(self, source_id: str, target_id: str) -> float:
        """Enhanced collaborative filtering"""
        source_users = [user for user, items in self.user_interactions.items()
                       if source_id in items]

        if not source_users:
            return 0.4

        target_likes = sum(1 for user in source_users
                          if target_id in self.user_interactions.get(user, []))

        if target_likes == 0:
            return 0.3

        collab_score = target_likes / len(source_users)

        if collab_score > 0.3:
            collab_score = min(1.0, collab_score * 1.4)
        elif collab_score > 0.1:
            collab_score = min(1.0, collab_score * 1.2)

        return collab_score

    def _calculate_quality_score(self, item: Dict) -> float:
        """Calculate item quality score"""
        quality = 0.0

        quality += item['popularity_score'] * 0.4
        quality += item.get('engagement_score', 0.5) * 0.3

        if len(item['themes']) >= 3:
            quality += 0.1
        if len(item['genres']) >= 2:
            quality += 0.1
        if len(item.get('description', '')) > 100:
            quality += 0.1

        return min(1.0, quality)

    def get_recommendations(self, input_title: str, num_recommendations: int = 10) -> Dict:
        """Enhanced recommendation API with higher confidence scores"""
        source_content = self.find_content(input_title)
        if not source_content:
            return {
                'success': False,
                'error': f"Content '{input_title}' not found",
                'available_platforms': list(self.platform_index.keys()),
                'total_content': len(self.content_database),
                'sample_titles': [item['title'] for item in self.content_database[:10]]
            }

        recommendations = []
        source_id = source_content['content_id']

        for item in self.content_database:
            if item['content_id'] == source_id:
                continue

            content_sim = self._calculate_enhanced_content_similarity(source_content, item)
            collab_score = self._calculate_enhanced_collaborative_score(source_id, item['content_id'])
            popularity_score = item['popularity_score']
            quality_score = self._calculate_quality_score(item)

            cross_platform_bonus = 0.2 if item['platform'] != source_content['platform'] else 0.1

            if quality_score < self.quality_threshold:
                continue

            base_score = (
                content_sim * self.weights['content_based'] +
                collab_score * self.weights['collaborative'] +
                popularity_score * self.weights['popularity'] +
                cross_platform_bonus * self.weights['cross_platform_bonus']
            )

            final_score = base_score + self.confidence_boost
            final_score *= (0.8 + quality_score * 0.4)

            final_score = max(0.5, min(0.99, final_score))

            confidence = max(self.min_confidence, min(99, int(final_score * 100)))

            reason = self._generate_enhanced_reason(
                source_content, item, content_sim, collab_score, quality_score
            )

            recommendations.append({
                'content_id': item['content_id'],
                'title': item['title'],
                'platform': item['platform'],
                'confidence_score': confidence,
                'genres': item['genres'],
                'themes': item['themes'],
                'moods': item['moods'],
                'description': item['description'],
                'popularity_score': item['popularity_score'],
                'quality_score': round(quality_score, 3),
                'reason': reason,
                'final_score': final_score,
                'content_similarity': round(content_sim, 3),
                'collaborative_score': round(collab_score, 3),
                'source': item.get('source', 'unknown')
            })

        recommendations.sort(key=lambda x: x['final_score'], reverse=True)

        filtered_recommendations = [
            rec for rec in recommendations
            if rec['confidence_score'] >= self.min_confidence
        ]

        top_recommendations = filtered_recommendations[:num_recommendations]

        if len(top_recommendations) < num_recommendations and recommendations:
            remaining_needed = num_recommendations - len(top_recommendations)
            additional_recs = recommendations[len(filtered_recommendations):][:remaining_needed]

            for rec in additional_recs:
                rec['confidence_score'] = max(self.min_confidence, rec['confidence_score'])
                rec['reason'] += " (Confidence boosted)"

            top_recommendations.extend(additional_recs)

        cross_platform_count = sum(1 for r in top_recommendations
                                 if r['platform'] != source_content['platform'])

        return {
            'success': True,
            'source': {
                'title': source_content['title'],
                'platform': source_content['platform'],
                'themes': source_content['themes'],
                'genres': source_content['genres'],
                'source': source_content.get('source', 'unknown')
            },
            'recommendations': top_recommendations,
            'summary': {
                'total_recommendations': len(top_recommendations),
                'cross_platform_count': cross_platform_count,
                'same_platform_count': len(top_recommendations) - cross_platform_count,
                'avg_confidence': round(np.mean([r['confidence_score'] for r in top_recommendations]), 1) if top_recommendations else 0,
                'min_confidence_threshold': self.min_confidence,
                'high_confidence_count': sum(1 for r in top_recommendations if r['confidence_score'] >= 75),
                'avg_quality_score': round(np.mean([r['quality_score'] for r in top_recommendations]), 3) if top_recommendations else 0,
                'data_sources': list(set(r['source'] for r in top_recommendations))
            },
            'metadata': {
                'total_content_available': len(self.content_database),
                'platforms': list(self.platform_index.keys()),
                'recommendation_weights': self.weights,
                'confidence_boost_applied': self.confidence_boost,
                'quality_filter_threshold': self.quality_threshold,
                'data_sources_loaded': list(self.csv_files.keys()) + (['google_sheets'] if self.google_sheets_url else [])
            }
        }

    def _generate_enhanced_reason(self, source: Dict, target: Dict,
                                content_sim: float, collab_score: float,
                                quality_score: float) -> str:
        """Generate enhanced recommendation reasons"""
        reasons = []

        common_themes = set(source['themes']) & set(target['themes'])
        common_genres = set(source['genres']) & set(target['genres'])
        common_moods = set(source['moods']) & set(target['moods'])

        if len(common_themes) >= 2:
            reasons.append(f"Strong thematic match: {', '.join(list(common_themes)[:2])}")
        elif common_themes:
            reasons.append(f"Similar themes: {', '.join(list(common_themes)[:2])}")

        if len(common_genres) >= 2:
            reasons.append(f"Multiple shared genres: {', '.join(list(common_genres)[:2])}")
        elif common_genres:
            reasons.append(f"Genre match: {', '.join(list(common_genres)[:1])}")

        if common_moods:
            reasons.append(f"Similar mood: {list(common_moods)[0]}")

        if collab_score > 0.4:
            reasons.append("Highly recommended by similar users")
        elif collab_score > 0.2:
            reasons.append("Popular with similar users")

        if quality_score > 0.8:
            reasons.append("Premium content")
        elif target['popularity_score'] > 0.8:
            reasons.append("Highly rated")

        if target['platform'] != source['platform']:
            reasons.append("Cross-platform discovery")

        if content_sim > 0.7:
            reasons.append("Excellent content match")
        elif content_sim > 0.5:
            reasons.append("Strong content similarity")

        if not reasons:
            reasons.append("Curated recommendation")

        return " • ".join(reasons[:4])

    def get_platform_stats(self) -> Dict:
        """Get statistics about loaded data"""
        stats = {
            'total_items': len(self.content_database),
            'platforms': {},
            'sources': {},
            'top_genres': {},
            'top_themes': {}
        }

        for item in self.content_database:
            platform = item['platform']
            source = item.get('source', 'unknown')

            stats['platforms'][platform] = stats['platforms'].get(platform, 0) + 1
            stats['sources'][source] = stats['sources'].get(source, 0) + 1

            for genre in item['genres']:
                stats['top_genres'][genre] = stats['top_genres'].get(genre, 0) + 1

            for theme in item['themes']:
                stats['top_themes'][theme] = stats['top_themes'].get(theme, 0) + 1

        # Sort by count
        stats['top_genres'] = dict(sorted(stats['top_genres'].items(),
                                        key=lambda x: x[1], reverse=True)[:10])
        stats['top_themes'] = dict(sorted(stats['top_themes'].items(),
                                        key=lambda x: x[1], reverse=True)[:10])

        return stats

    def search_content(self, query: str, platform: str = None, limit: int = 20) -> List[Dict]:
        """Search for content by title, genre, or theme"""
        query = query.lower().strip()
        results = []

        for item in self.content_database:
            if platform and item['platform'] != platform:
                continue

            match_score = 0

            # Title match
            if query in item['title'].lower():
                match_score += 10

            # Genre match
            if any(query in genre.lower() for genre in item['genres']):
                match_score += 5

            # Theme match
            if any(query in theme.lower() for theme in item['themes']):
                match_score += 5

            # Description match
            if query in item.get('description', '').lower():
                match_score += 2

            if match_score > 0:
                item_copy = item.copy()
                item_copy['match_score'] = match_score
                results.append(item_copy)

        results.sort(key=lambda x: x['match_score'], reverse=True)
        return results[:limit]


# Usage functions
def initialize_recommendation_service_from_csvs(csv_files: Dict[str, str] = None) -> ContentRecommendationService:
    """Initialize the recommendation service with CSV files"""
    default_csvs = {
        'anime': 'anime_reddit.csv',
        'movie_reddit': 'movie_reddit.csv',
        'movie_tmdb': 'movie_tmdb.csv',
        'game_reddit': 'video_game_reddit.csv',
        'game_twitch': 'video_game_twitch.csv'
    }

    service = ContentRecommendationService(csv_files=csv_files or default_csvs)
    return service

def initialize_recommendation_service_from_sheets(google_sheets_url: str,
                                                csv_files: Dict[str, str] = None) -> ContentRecommendationService:
    """Initialize the recommendation service with Google Sheets and optional CSV files"""
    service = ContentRecommendationService(csv_files=csv_files, google_sheets_url=google_sheets_url)
    return service

def initialize_recommendation_service_hybrid(google_sheets_url: str = None,
                                           csv_files: Dict[str, str] = None) -> ContentRecommendationService:
    """Initialize with both CSV files and Google Sheets"""
    sheets_url = google_sheets_url or "https://docs.google.com/spreadsheets/d/1wu53RWdFsxgq6DfglcPTmkxI7QPSw8mTjjAm_1oFmQk/edit?usp=sharing"

    default_csvs = {
        'anime': 'anime_reddit.csv',
        'movie_reddit': 'movie_reddit.csv',
        'movie_tmdb': 'movie_tmdb.csv',
        'game_reddit': 'video_game_reddit.csv',
        'game_twitch': 'video_game_twitch.csv'
    }

    service = ContentRecommendationService(
        csv_files=csv_files or default_csvs,
        google_sheets_url=sheets_url
    )
    return service


# Example usage and testing
if __name__ == "__main__":
    print("Initializing Enhanced Recommendation Service with new data sources...")

    # Option 1: Use only CSV files
    print("\n=== Option 1: CSV Files Only ===")
    service_csv = initialize_recommendation_service_from_csvs()
    print(f"CSV Service - Total content loaded: {len(service_csv.content_database)}")
    stats_csv = service_csv.get_platform_stats()
    print(f"Platforms: {stats_csv['platforms']}")
    print(f"Sources: {stats_csv['sources']}")

    # Option 2: Use Google Sheets
    print("\n=== Option 2: Google Sheets + CSV Files ===")
    service_hybrid = initialize_recommendation_service_hybrid()
    print(f"Hybrid Service - Total content loaded: {len(service_hybrid.content_database)}")
    stats_hybrid = service_hybrid.get_platform_stats()
    print(f"Platforms: {stats_hybrid['platforms']}")
    print(f"Sources: {stats_hybrid['sources']}")

    # Test recommendations
    print("\n=== Testing Recommendations ===")
    if service_hybrid.content_database:
        sample_title = service_hybrid.content_database[0]['title']
        print(f"Getting recommendations for: {sample_title}")

        result = service_hybrid.get_recommendations(sample_title, 5)

        if result['success']:
            print(f"Source: {result['source']['title']} ({result['source']['platform']})")
            print(f"Average confidence: {result['summary']['avg_confidence']}%")
            print(f"Data sources in results: {result['summary']['data_sources']}")

            print("\nTop recommendations:")
            for i, rec in enumerate(result['recommendations'], 1):
                print(f"{i}. {rec['title']} ({rec['platform']}) - {rec['confidence_score']}%")
                print(f"   Source: {rec['source']} | Reason: {rec['reason']}")
        else:
            print(f"Error: {result['error']}")

    # Test search functionality
    print("\n=== Testing Search ===")
    search_results = service_hybrid.search_content("action", limit=5)
    print(f"Search results for 'action': {len(search_results)} found")
    for result in search_results[:3]:
        print(f"- {result['title']} ({result['platform']}) - Match score: {result['match_score']}")

    print("\n=== Data Summary ===")
    print(f"Total items loaded: {stats_hybrid['total_items']}")
    print(f"Top genres: {list(stats_hybrid['top_genres'].keys())[:5]}")
    print(f"Top themes: {list(stats_hybrid['top_themes'].keys())[:5]}")
    print(f"Available data sources: {stats_hybrid['sources']}")
    print("\nRecommendation service ready!")

Initializing Enhanced Recommendation Service with new data sources...

=== Option 1: CSV Files Only ===
CSV Service - Total content loaded: 3295
Platforms: {'anime': 635, 'movie': 1068, 'game': 1592}
Sources: {'reddit': 635, 'movie_reddit': 920, 'movie_tmdb': 148, 'game_reddit': 1542, 'game_twitch': 50}

=== Option 2: Google Sheets + CSV Files ===
Hybrid Service - Total content loaded: 3930
Platforms: {'anime': 635, 'movie': 1068, 'game': 1592, 'reddit': 635}
Sources: {'reddit': 635, 'movie_reddit': 920, 'movie_tmdb': 148, 'game_reddit': 1542, 'game_twitch': 50, 'google_sheets': 635}

=== Testing Recommendations ===
Getting recommendations for: Anime Questions, Recommendations, and Discussion - September 16, 2025
Source: Anime Questions, Recommendations, and Discussion - September 16, 2025 (reddit)
Average confidence: 64.2%
Data sources in results: ['movie_tmdb']

Top recommendations:
1. The Science Adventures of Tom and Huck (movie) - 66%
   Source: movie_tmdb | Reason: Popular with s

## *Dashboard*

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
import json
import logging
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import requests
from io import StringIO
import re
from typing import Dict, List, Tuple, Optional
import time

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Pre-configured API credentials from documentation
REDDIT_CONFIG = {
    'CLIENT_ID': "YOUR_CLIENT_ID",
    'CLIENT_SECRET': "YOUR_CLIENT_SECRET",
    'USER_AGENT': 'YOUR_USER_AGENT'
}

TWITCH_CONFIG = {
    'CLIENT_ID': 'YOUR_CLIENT_ID',
    'CLIENT_SECRET': 'YOUR_CLIENT_SECRET'
}

TMDB_CONFIG = {
    'API_KEY': 'YOUR_API_KEY'
}

GOOGLE_SHEETS_CONFIG = {
    'SHEET_ID': 'YOUR_SHEET_ID',
    'SHEET_URL': 'YOUR_SHEET_URL'
}

# Target subreddits from documentation
TARGET_SUBREDDITS = {
    'general': ['anime', 'gaming', 'movies'],
    'gaming_subgenres': ['JRPG', 'visualnovels'],
    'recommendation_focused': ['MovieSuggestions', 'gamingsuggestions'],
    'cross_reference': ['tipofmytongue']
}

class CrossPlatformRecommendationService:
    """Enhanced recommendation service based on documentation specifications"""

    def __init__(self):
        self.content_database = []
        self.platform_index = {}
        self.genre_index = {}
        self.theme_index = {}
        self.subreddit_index = {}
        self.user_interactions = {}

        # Enhanced recommendation weights
        self.weights = {
            'content_based': 0.45,
            'collaborative': 0.25,
            'popularity': 0.20,
            'cross_platform_bonus': 0.10
        }

        # Initialize with sample data and try to load from Google Sheets
        self._initialize_data()
        self._build_indexes()
        self._simulate_collaborative_data()

    def _initialize_data(self):
        """Initialize with expanded sample data and attempt to load from Google Sheets"""
        # Create comprehensive sample data based on documentation
        self.content_database = [
            # Anime entries
            {
                'content_id': 'anime_aot',
                'title': 'Attack on Titan',
                'platform': 'anime',
                'genres': ['action', 'drama', 'dark fantasy'],
                'themes': ['survival', 'war_conflict', 'moral_ambiguity', 'friendship'],
                'moods': ['dark', 'intense', 'emotional'],
                'popularity_score': 0.95,
                'engagement_score': 0.92,
                'description': 'Humanity fights for survival against giant humanoid Titans in a post-apocalyptic world.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'young_adult',
                'art_style': 'dark_realistic',
                'pacing': 'intense'
            },
            {
                'content_id': 'anime_demon_slayer',
                'title': 'Demon Slayer',
                'platform': 'anime',
                'genres': ['action', 'supernatural', 'shounen'],
                'themes': ['family', 'revenge', 'supernatural', 'friendship'],
                'moods': ['intense', 'emotional', 'heroic'],
                'popularity_score': 0.92,
                'engagement_score': 0.90,
                'description': 'A boy becomes a demon slayer to save his sister and avenge his family.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'teen',
                'art_style': 'vibrant_traditional',
                'pacing': 'fast'
            },
            {
                'content_id': 'anime_steins_gate',
                'title': 'Steins;Gate',
                'platform': 'anime',
                'genres': ['sci-fi', 'thriller', 'drama'],
                'themes': ['time_travel', 'science', 'friendship', 'sacrifice'],
                'moods': ['mysterious', 'emotional', 'intense'],
                'popularity_score': 0.94,
                'engagement_score': 0.93,
                'description': 'Time travel thriller exploring consequences of changing the past.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'young_adult',
                'art_style': 'modern_anime',
                'pacing': 'slow_build'
            },
            {
                'content_id': 'anime_death_note',
                'title': 'Death Note',
                'platform': 'anime',
                'genres': ['psychological', 'supernatural', 'thriller'],
                'themes': ['power_struggle', 'moral_ambiguity', 'justice', 'identity'],
                'moods': ['dark', 'philosophical', 'intense'],
                'popularity_score': 0.91,
                'engagement_score': 0.89,
                'description': 'A student discovers a notebook that can kill anyone whose name is written in it.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'young_adult',
                'art_style': 'gothic_realistic',
                'pacing': 'moderate'
            },
            {
                'content_id': 'anime_fullmetal',
                'title': 'Fullmetal Alchemist: Brotherhood',
                'platform': 'anime',
                'genres': ['action', 'adventure', 'dark fantasy'],
                'themes': ['family', 'sacrifice', 'war_conflict', 'redemption'],
                'moods': ['emotional', 'intense', 'epic'],
                'popularity_score': 0.96,
                'engagement_score': 0.95,
                'description': 'Two brothers use alchemy to search for the Philosophers Stone after a disastrous attempt to resurrect their mother.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'teen',
                'art_style': 'traditional_anime',
                'pacing': 'moderate'
            },
            {
                'content_id': 'anime_spirited_away',
                'title': 'Spirited Away',
                'platform': 'anime',
                'genres': ['adventure', 'family', 'supernatural'],
                'themes': ['coming_of_age', 'family', 'supernatural', 'identity'],
                'moods': ['magical', 'emotional', 'mysterious'],
                'popularity_score': 0.93,
                'engagement_score': 0.91,
                'description': 'A young girl enters a world ruled by gods and witches where humans are turned into beasts.',
                'source': 'reddit',
                'subreddit': 'anime',
                'demographic': 'all_ages',
                'art_style': 'studio_ghibli',
                'pacing': 'moderate'
            },
            # Game entries
            {
                'content_id': 'game_last_of_us',
                'title': 'The Last of Us',
                'platform': 'game',
                'genres': ['survival horror', 'action', 'adventure'],
                'themes': ['survival', 'family', 'moral_ambiguity', 'loss'],
                'moods': ['dark', 'emotional', 'intense'],
                'popularity_score': 0.89,
                'engagement_score': 0.91,
                'description': 'Post-apocalyptic survival game focusing on human relationships in a zombie-infested world.',
                'source': 'reddit',
                'subreddit': 'gaming',
                'demographic': 'mature',
                'art_style': 'photorealistic',
                'pacing': 'moderate'
            },
            {
                'content_id': 'game_god_of_war',
                'title': 'God of War',
                'platform': 'game',
                'genres': ['action', 'adventure', 'mythology'],
                'themes': ['family', 'redemption', 'mythology', 'power_struggle'],
                'moods': ['emotional', 'intense', 'epic'],
                'popularity_score': 0.88,
                'engagement_score': 0.89,
                'description': 'Norse mythology adventure focusing on father-son relationship and redemption.',
                'source': 'reddit',
                'subreddit': 'gaming',
                'demographic': 'mature',
                'art_style': 'cinematic_realistic',
                'pacing': 'moderate'
            },
            {
                'content_id': 'game_nier_automata',
                'title': 'NieR: Automata',
                'platform': 'game',
                'genres': ['action rpg', 'sci-fi', 'philosophical'],
                'themes': ['identity', 'philosophy', 'war_conflict', 'existentialism'],
                'moods': ['philosophical', 'emotional', 'mysterious'],
                'popularity_score': 0.85,
                'engagement_score': 0.88,
                'description': 'Philosophical action RPG exploring themes of consciousness and humanity through android protagonists.',
                'source': 'reddit',
                'subreddit': 'JRPG',
                'demographic': 'young_adult',
                'art_style': 'anime_realistic',
                'pacing': 'moderate'
            },
            {
                'content_id': 'game_witcher3',
                'title': 'The Witcher 3: Wild Hunt',
                'platform': 'game',
                'genres': ['rpg', 'fantasy', 'adventure'],
                'themes': ['family', 'moral_ambiguity', 'war_conflict', 'love_romance'],
                'moods': ['dark', 'emotional', 'epic'],
                'popularity_score': 0.92,
                'engagement_score': 0.91,
                'description': 'Fantasy RPG following Geralt of Rivia as he searches for his adopted daughter.',
                'source': 'reddit',
                'subreddit': 'gaming',
                'demographic': 'mature',
                'art_style': 'fantasy_realistic',
                'pacing': 'slow_build'
            },
            {
                'content_id': 'game_persona5',
                'title': 'Persona 5',
                'platform': 'game',
                'genres': ['jrpg', 'supernatural', 'social simulation'],
                'themes': ['friendship', 'justice', 'identity', 'coming_of_age'],
                'moods': ['stylish', 'emotional', 'mysterious'],
                'popularity_score': 0.87,
                'engagement_score': 0.88,
                'description': 'Stylish JRPG about phantom thieves changing corrupt hearts in modern Tokyo.',
                'source': 'reddit',
                'subreddit': 'JRPG',
                'demographic': 'teen',
                'art_style': 'anime_stylized',
                'pacing': 'slow_build'
            },
            {
                'content_id': 'game_cyberpunk',
                'title': 'Cyberpunk 2077',
                'platform': 'game',
                'genres': ['rpg', 'sci-fi', 'action'],
                'themes': ['identity', 'technology', 'power_struggle', 'moral_ambiguity'],
                'moods': ['dark', 'intense', 'philosophical'],
                'popularity_score': 0.78,
                'engagement_score': 0.82,
                'description': 'Futuristic RPG set in Night City where technology and humanity collide.',
                'source': 'reddit',
                'subreddit': 'gaming',
                'demographic': 'mature',
                'art_style': 'cyberpunk_realistic',
                'pacing': 'fast'
            },
            # Movie entries
            {
                'content_id': 'movie_mad_max',
                'title': 'Mad Max: Fury Road',
                'platform': 'movie',
                'genres': ['action', 'sci-fi', 'adventure'],
                'themes': ['survival', 'power_struggle', 'justice'],
                'moods': ['intense', 'dark', 'thrilling'],
                'popularity_score': 0.87,
                'engagement_score': 0.85,
                'description': 'Post-apocalyptic action film with high-octane chase sequences and strong female protagonist.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'adult',
                'art_style': 'gritty_realistic',
                'pacing': 'very_fast'
            },
            {
                'content_id': 'movie_john_wick',
                'title': 'John Wick',
                'platform': 'movie',
                'genres': ['action', 'thriller', 'crime'],
                'themes': ['revenge', 'loss', 'justice'],
                'moods': ['intense', 'dark', 'stylish'],
                'popularity_score': 0.83,
                'engagement_score': 0.84,
                'description': 'Retired hitman seeks vengeance for his dog in stylized action sequences.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'adult',
                'art_style': 'stylized_realistic',
                'pacing': 'fast'
            },
            {
                'content_id': 'movie_blade_runner',
                'title': 'Blade Runner 2049',
                'platform': 'movie',
                'genres': ['sci-fi', 'neo-noir', 'thriller'],
                'themes': ['identity', 'philosophy', 'technology', 'existentialism'],
                'moods': ['philosophical', 'dark', 'mysterious'],
                'popularity_score': 0.86,
                'engagement_score': 0.87,
                'description': 'Neo-noir sci-fi exploring what it means to be human in a world of replicants.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'adult',
                'art_style': 'neo_noir_realistic',
                'pacing': 'slow_build'
            },
            {
                'content_id': 'movie_parasite',
                'title': 'Parasite',
                'platform': 'movie',
                'genres': ['thriller', 'dark comedy', 'drama'],
                'themes': ['power_struggle', 'moral_ambiguity', 'family', 'class_conflict'],
                'moods': ['dark', 'intense', 'emotional'],
                'popularity_score': 0.91,
                'engagement_score': 0.90,
                'description': 'Dark thriller exploring class conflict through the story of two families.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'mature',
                'art_style': 'realistic_contemporary',
                'pacing': 'moderate'
            },
            {
                'content_id': 'movie_interstellar',
                'title': 'Interstellar',
                'platform': 'movie',
                'genres': ['sci-fi', 'drama', 'adventure'],
                'themes': ['family', 'sacrifice', 'time_travel', 'survival'],
                'moods': ['emotional', 'epic', 'philosophical'],
                'popularity_score': 0.88,
                'engagement_score': 0.89,
                'description': 'Epic sci-fi about a father who travels through space and time to save humanity.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'adult',
                'art_style': 'epic_realistic',
                'pacing': 'slow_build'
            },
            {
                'content_id': 'movie_your_name',
                'title': 'Your Name',
                'platform': 'movie',
                'genres': ['animation', 'romance', 'supernatural'],
                'themes': ['love_romance', 'time_travel', 'identity', 'supernatural'],
                'moods': ['romantic', 'emotional', 'magical'],
                'popularity_score': 0.89,
                'engagement_score': 0.87,
                'description': 'Anime film about two teenagers who mysteriously swap bodies.',
                'source': 'tmdb',
                'subreddit': 'movies',
                'demographic': 'teen',
                'art_style': 'anime_cinematic',
                'pacing': 'moderate'
            }
        ]

        # Try to load from Google Sheets
        try:
            self._load_from_google_sheets()
        except Exception as e:
            logger.warning(f"Could not load from Google Sheets: {e}")

    def _load_from_google_sheets(self):
        """Load additional data from Google Sheets"""
        try:
            csv_url = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SHEETS_CONFIG['SHEET_ID']}/export?format=csv"
            response = requests.get(csv_url, timeout=10)
            response.raise_for_status()

            df = pd.read_csv(StringIO(response.text))
            logger.info(f"Loaded {len(df)} rows from Google Sheets")

            # Process Google Sheets data
            sheets_data = self._process_sheets_data(df)
            self.content_database.extend(sheets_data)

        except Exception as e:
            logger.error(f"Error loading from Google Sheets: {e}")

    def _process_sheets_data(self, df: pd.DataFrame) -> List[Dict]:
        """Process data from Google Sheets"""
        items = []

        for idx, row in df.iterrows():
            try:
                # Extract title
                title_cols = ['title', 'name', 'content_name']
                title = None
                for col in title_cols:
                    if col in df.columns and pd.notna(row[col]):
                        title = str(row[col]).strip()
                        break

                if not title:
                    continue

                # Extract platform
                platform = str(row.get('platform', 'unknown')).lower()

                # Extract other fields
                genres = self._parse_list_field(str(row.get('genres', '')))
                themes = self._parse_list_field(str(row.get('themes', '')))

                score = float(row.get('score', 0.5))
                if score > 10:
                    popularity_score = min(1.0, score / 100.0)
                else:
                    popularity_score = min(1.0, score / 10.0)

                item = {
                    'content_id': f'sheets_{platform}_{idx}',
                    'title': title,
                    'platform': platform,
                    'genres': genres,
                    'themes': themes,
                    'moods': self._extract_moods(genres, themes, ''),
                    'popularity_score': popularity_score,
                    'engagement_score': popularity_score * 0.95,
                    'description': str(row.get('description', ''))[:300],
                    'source': 'google_sheets',
                    'subreddit': str(row.get('subreddit', 'unknown')),
                    'demographic': str(row.get('demographic', 'general')),
                    'art_style': str(row.get('art_style', 'standard')),
                    'pacing': str(row.get('pacing', 'moderate'))
                }

                items.append(item)

            except Exception as e:
                logger.warning(f"Error processing sheets row {idx}: {e}")
                continue

        return items

    def _parse_list_field(self, field_value: str) -> List[str]:
        """Parse list-like string fields"""
        if not field_value or field_value == 'nan':
            return []

        field_value = str(field_value).strip().lower()

        # Handle JSON-like lists
        if field_value.startswith('[') and field_value.endswith(']'):
            try:
                clean_value = field_value.replace("'", '"')
                items = json.loads(clean_value)
                return [str(item).strip() for item in items if str(item).strip()]
            except:
                field_value = field_value[1:-1]

        # Handle common delimiters
        for delimiter in [',', '|', ';', '/']:
            if delimiter in field_value:
                items = [item.strip() for item in field_value.split(delimiter)]
                return [item for item in items if item and item != 'nan']

        return [field_value.strip()] if field_value.strip() else []

    def _extract_moods(self, genres: List[str], themes: List[str], description: str) -> List[str]:
        """Extract emotional mood/tone"""
        moods = set()

        genre_moods = {
            'horror': 'dark',
            'thriller': 'intense',
            'action': 'intense',
            'drama': 'emotional',
            'comedy': 'light_hearted',
            'romance': 'romantic',
            'sci-fi': 'philosophical',
            'mystery': 'mysterious',
            'war': 'dark',
            'crime': 'dark',
            'fantasy': 'magical',
            'adventure': 'exciting'
        }

        for genre in genres:
            if genre in genre_moods:
                moods.add(genre_moods[genre])

        theme_moods = {
            'survival': 'dark',
            'war_conflict': 'dark',
            'moral_ambiguity': 'philosophical',
            'power_struggle': 'intense',
            'revenge': 'dark',
            'family': 'emotional',
            'friendship': 'emotional',
            'love': 'romantic'
        }

        for theme in themes:
            if theme in theme_moods:
                moods.add(theme_moods[theme])

        return list(moods) if moods else ['neutral']

    def _build_indexes(self):
        """Build fast lookup indexes"""
        self.platform_index = {}
        self.genre_index = {}
        self.theme_index = {}
        self.subreddit_index = {}

        for item in self.content_database:
            # Platform index
            platform = item['platform']
            if platform not in self.platform_index:
                self.platform_index[platform] = []
            self.platform_index[platform].append(item)

            # Genre index
            for genre in item['genres']:
                if genre not in self.genre_index:
                    self.genre_index[genre] = []
                self.genre_index[genre].append(item)

            # Theme index
            for theme in item['themes']:
                if theme not in self.theme_index:
                    self.theme_index[theme] = []
                self.theme_index[theme].append(item)

            # Subreddit index
            subreddit = item.get('subreddit', 'unknown')
            if subreddit not in self.subreddit_index:
                self.subreddit_index[subreddit] = []
            self.subreddit_index[subreddit].append(item)

    def _simulate_collaborative_data(self):
        """Generate simulated user interaction data based on documentation"""
        import random
        random.seed(42)
        self.user_interactions = {}

        # Create user profiles based on cross-platform consumption patterns
        for i in range(150):
            user_items = []

            # Simulate cross-platform preferences
            if random.random() < 0.3:  # 30% cross-platform users
                platforms = random.sample(list(self.platform_index.keys()), random.randint(2, 3))
                for platform in platforms:
                    platform_items = self.platform_index.get(platform, [])
                    if platform_items:
                        num_items = random.randint(1, 3)
                        selected = random.sample(platform_items, min(num_items, len(platform_items)))
                        user_items.extend([item['content_id'] for item in selected])
            else:  # Single platform users
                platform = random.choice(list(self.platform_index.keys()))
                platform_items = self.platform_index.get(platform, [])
                if platform_items:
                    num_items = random.randint(2, 5)
                    selected = random.sample(platform_items, min(num_items, len(platform_items)))
                    user_items = [item['content_id'] for item in selected]

            if user_items:
                self.user_interactions[f'user_{i}'] = user_items

    def find_content(self, title: str) -> Optional[Dict]:
        """Find content by title with fuzzy matching"""
        title_lower = title.lower().strip()

        # Exact match first
        for item in self.content_database:
            if item['title'].lower() == title_lower:
                return item

        # Partial match
        for item in self.content_database:
            if title_lower in item['title'].lower() or item['title'].lower() in title_lower:
                return item

        return None

    def get_recommendations(self, input_title: str, num_recommendations: int = 10,
                          platform_filter: str = None, mood_filter: str = None) -> Dict:
        """Get enhanced recommendations with filters"""
        source_content = self.find_content(input_title)
        if not source_content:
            available_titles = [item['title'] for item in self.content_database[:20]]
            return {
                'success': False,
                'error': f"Content '{input_title}' not found",
                'available_titles': available_titles,
                'suggestions': "Try: " + ", ".join(available_titles[:5])
            }

        all_recommendations = []
        source_id = source_content['content_id']

        # First, calculate scores for ALL items (no filtering yet)
        for item in self.content_database:
            if item['content_id'] == source_id:
                continue

            # Calculate similarity scores
            content_sim = self._calculate_content_similarity(source_content, item)
            collab_score = self._calculate_collaborative_score(source_id, item['content_id'])
            popularity_score = item['popularity_score']

            # Cross-platform bonus
            cross_platform_bonus = 0.15 if item['platform'] != source_content['platform'] else 0.05

            # Calculate final score
            final_score = (
                content_sim * self.weights['content_based'] +
                collab_score * self.weights['collaborative'] +
                popularity_score * self.weights['popularity'] +
                cross_platform_bonus * self.weights['cross_platform_bonus']
            )

            confidence = max(50, min(99, int(final_score * 100 + np.random.uniform(5, 15))))

            reason = self._generate_reason(source_content, item, content_sim)

            all_recommendations.append({
                'content_id': item['content_id'],
                'title': item['title'],
                'platform': item['platform'],
                'confidence_score': confidence,
                'genres': item['genres'],
                'themes': item['themes'],
                'moods': item['moods'],
                'description': item['description'],
                'reason': reason,
                'source': item['source'],
                'subreddit': item.get('subreddit', 'unknown'),
                'demographic': item.get('demographic', 'general'),
                'art_style': item.get('art_style', 'standard'),
                'final_score': final_score
            })

        # Sort all recommendations by score
        all_recommendations.sort(key=lambda x: x['final_score'], reverse=True)

        # NOW apply filters to get the requested number
        filtered_recommendations = []
        for rec in all_recommendations:
            # Check platform filter
            if platform_filter and platform_filter != "All" and rec['platform'] != platform_filter.lower():
                continue

            # Check mood filter
            if mood_filter and mood_filter != "All" and mood_filter.lower() not in rec['moods']:
                continue

            # Add to filtered list
            filtered_recommendations.append(rec)

            # Stop when we have enough recommendations
            if len(filtered_recommendations) >= num_recommendations:
                break

        # If we don't have enough filtered results, pad with top unfiltered results
        if len(filtered_recommendations) < num_recommendations:
            remaining_needed = num_recommendations - len(filtered_recommendations)
            filtered_ids = {rec['content_id'] for rec in filtered_recommendations}

            for rec in all_recommendations:
                if rec['content_id'] not in filtered_ids:
                    filtered_recommendations.append(rec)
                    remaining_needed -= 1
                    if remaining_needed <= 0:
                        break

        top_recommendations = filtered_recommendations[:num_recommendations]

        # Calculate summary statistics
        cross_platform_count = sum(1 for r in top_recommendations
                                 if r['platform'] != source_content['platform'])

        return {
            'success': True,
            'source': {
                'title': source_content['title'],
                'platform': source_content['platform'],
                'themes': source_content['themes'],
                'genres': source_content['genres'],
                'subreddit': source_content.get('subreddit', 'unknown')
            },
            'recommendations': top_recommendations,
            'summary': {
                'total_recommendations': len(top_recommendations),
                'cross_platform_count': cross_platform_count,
                'same_platform_count': len(top_recommendations) - cross_platform_count,
                'avg_confidence': round(np.mean([r['confidence_score'] for r in top_recommendations]), 1) if top_recommendations else 0,
                'platforms_covered': list(set(r['platform'] for r in top_recommendations)),
                'subreddits_covered': list(set(r['subreddit'] for r in top_recommendations))
            }
        }

    def _calculate_content_similarity(self, item1: Dict, item2: Dict) -> float:
        """Calculate content-based similarity"""
        # Genre similarity
        genres1, genres2 = set(item1['genres']), set(item2['genres'])
        genre_sim = len(genres1 & genres2) / max(len(genres1 | genres2), 1) if genres1 or genres2 else 0

        # Theme similarity
        themes1, themes2 = set(item1['themes']), set(item2['themes'])
        theme_sim = len(themes1 & themes2) / max(len(themes1 | themes2), 1) if themes1 or themes2 else 0

        # Mood similarity
        moods1, moods2 = set(item1['moods']), set(item2['moods'])
        mood_sim = len(moods1 & moods2) / max(len(moods1 | moods2), 1) if moods1 or moods2 else 0

        # Demographic similarity
        demo_sim = 1.0 if item1.get('demographic') == item2.get('demographic') else 0.3

        # Overall content similarity
        content_sim = (
            theme_sim * 0.4 +
            genre_sim * 0.3 +
            mood_sim * 0.2 +
            demo_sim * 0.1
        )

        return content_sim

    def _calculate_collaborative_score(self, source_id: str, target_id: str) -> float:
        """Calculate collaborative filtering score"""
        source_users = [user for user, items in self.user_interactions.items()
                       if source_id in items]

        if not source_users:
            return 0.4

        target_likes = sum(1 for user in source_users
                          if target_id in self.user_interactions.get(user, []))

        if target_likes == 0:
            return 0.3

        return min(1.0, target_likes / len(source_users) * 1.5)

    def _generate_reason(self, source: Dict, target: Dict, similarity: float) -> str:
        """Generate recommendation reason"""
        reasons = []

        common_themes = set(source['themes']) & set(target['themes'])
        common_genres = set(source['genres']) & set(target['genres'])
        common_moods = set(source['moods']) & set(target['moods'])

        if common_themes:
            reasons.append(f"Shared themes: {', '.join(list(common_themes)[:2])}")

        if common_genres:
            reasons.append(f"Similar genres: {', '.join(list(common_genres)[:2])}")

        if common_moods:
            reasons.append(f"Similar mood: {list(common_moods)[0]}")

        if target['platform'] != source['platform']:
            reasons.append("Cross-platform discovery")

        if target.get('subreddit') == source.get('subreddit'):
            reasons.append(f"Popular in r/{target.get('subreddit')}")

        if similarity > 0.7:
            reasons.append("Strong content match")

        return " • ".join(reasons[:3]) if reasons else "Curated recommendation"

    def get_platform_stats(self) -> Dict:
        """Get comprehensive platform statistics"""
        stats = {
            'total_items': len(self.content_database),
            'platforms': {},
            'sources': {},
            'subreddits': {},
            'demographics': {},
            'top_genres': {},
            'top_themes': {},
            'mood_distribution': {}
        }

        for item in self.content_database:
            # Platform counts
            platform = item['platform']
            stats['platforms'][platform] = stats['platforms'].get(platform, 0) + 1

            # Source counts
            source = item.get('source', 'unknown')
            stats['sources'][source] = stats['sources'].get(source, 0) + 1

            # Subreddit counts
            subreddit = item.get('subreddit', 'unknown')
            stats['subreddits'][subreddit] = stats['subreddits'].get(subreddit, 0) + 1

            # Demographics
            demographic = item.get('demographic', 'general')
            stats['demographics'][demographic] = stats['demographics'].get(demographic, 0) + 1

            # Genres
            for genre in item['genres']:
                stats['top_genres'][genre] = stats['top_genres'].get(genre, 0) + 1

            # Themes
            for theme in item['themes']:
                stats['top_themes'][theme] = stats['top_themes'].get(theme, 0) + 1

            # Moods
            for mood in item['moods']:
                stats['mood_distribution'][mood] = stats['mood_distribution'].get(mood, 0) + 1

        # Sort by count
        stats['top_genres'] = dict(sorted(stats['top_genres'].items(), key=lambda x: x[1], reverse=True)[:15])
        stats['top_themes'] = dict(sorted(stats['top_themes'].items(), key=lambda x: x[1], reverse=True)[:15])
        stats['mood_distribution'] = dict(sorted(stats['mood_distribution'].items(), key=lambda x: x[1], reverse=True))

        return stats

    def search_content(self, query: str, platform: str = None, subreddit: str = None, limit: int = 20) -> List[Dict]:
        """Enhanced search with multiple filters"""
        query = query.lower().strip()
        results = []

        for item in self.content_database:
            # Apply filters
            if platform and platform != "All" and item['platform'] != platform.lower():
                continue

            if subreddit and subreddit != "All" and item.get('subreddit') != subreddit.lower():
                continue

            match_score = 0

            # Title match (highest priority)
            if query in item['title'].lower():
                match_score += 10

            # Genre match
            if any(query in genre.lower() for genre in item['genres']):
                match_score += 5

            # Theme match
            if any(query in theme.lower() for theme in item['themes']):
                match_score += 4

            # Mood match
            if any(query in mood.lower() for mood in item['moods']):
                match_score += 3

            # Description match
            if query in item.get('description', '').lower():
                match_score += 2

            if match_score > 0:
                item_copy = item.copy()
                item_copy['match_score'] = match_score
                results.append(item_copy)

        results.sort(key=lambda x: x['match_score'], reverse=True)
        return results[:limit]

# Initialize the service
service = CrossPlatformRecommendationService()

def create_main_recommendation_interface():
    """Create the main recommendation interface"""

    def get_recommendations_with_filters(title, num_recs, platform_filter, mood_filter):
        if not title.strip():
            return "Please enter a title", None, None, ""

        result = service.get_recommendations(
            title.strip(),
            int(num_recs),
            platform_filter,
            mood_filter
        )

        if not result['success']:
            error_msg = f"Error: {result['error']}\n\n"
            if 'suggestions' in result:
                error_msg += f"Suggestions: {result['suggestions']}"
            return error_msg, None, None, ""

        # Format recommendations text
        recs_text = f" SOURCE: {result['source']['title']} ({result['source']['platform']})\n"
        recs_text += f" Subreddit: r/{result['source']['subreddit']}\n"
        recs_text += f" Themes: {', '.join(result['source']['themes'])}\n"
        recs_text += f" Genres: {', '.join(result['source']['genres'])}\n\n"
        recs_text += " RECOMMENDATIONS:\n" + "="*80 + "\n\n"

        for i, rec in enumerate(result['recommendations'], 1):
            recs_text += f"{i:2d}. {rec['title']} ({rec['platform'].upper()})\n"
            recs_text += f"    Confidence: {rec['confidence_score']}% | r/{rec['subreddit']}\n"
            recs_text += f"    Reason: {rec['reason']}\n"
            recs_text += f"    {rec['description'][:100]}{'...' if len(rec['description']) > 100 else ''}\n\n"

        # Create visualizations
        recommendations = result['recommendations']

        # Platform distribution
        platforms = [rec['platform'] for rec in recommendations]
        platform_counts = Counter(platforms)
        platform_fig = px.pie(
            values=list(platform_counts.values()),
            names=list(platform_counts.keys()),
            title="Recommendations by Platform",
            color_discrete_sequence=px.colors.qualitative.Set3
        )

        # Confidence scores
        titles = [rec['title'][:15] + "..." if len(rec['title']) > 15 else rec['title'] for rec in recommendations]
        confidence_scores = [rec['confidence_score'] for rec in recommendations]
        confidence_fig = px.bar(
            x=confidence_scores,
            y=titles,
            orientation='h',
            title="Recommendation Confidence Scores",
            labels={'x': 'Confidence %', 'y': 'Content'},
            color=confidence_scores,
            color_continuous_scale="viridis"
        )

        # Summary text
        summary = result['summary']
        summary_text = f"""
📈 SUMMARY STATISTICS:
• Total Recommendations: {summary['total_recommendations']}
• Cross-Platform: {summary['cross_platform_count']}
• Same Platform: {summary['same_platform_count']}
• Average Confidence: {summary['avg_confidence']}%
• Platforms: {', '.join(summary['platforms_covered'])}
• Subreddits: {', '.join(summary['subreddits_covered'])}
        """

        return recs_text, platform_fig, confidence_fig, summary_text.strip()

    return get_recommendations_with_filters

def create_analytics_dashboard():
    """Create comprehensive analytics dashboard"""

    def generate_analytics():
        stats = service.get_platform_stats()

        # Platform distribution
        platform_fig = px.pie(
            values=list(stats['platforms'].values()),
            names=list(stats['platforms'].keys()),
            title="Content Distribution by Platform"
        )

        # Subreddit distribution
        subreddit_fig = px.bar(
            x=list(stats['subreddits'].values()),
            y=list(stats['subreddits'].keys()),
            orientation='h',
            title="Content by Subreddit",
            labels={'x': 'Count', 'y': 'Subreddit'}
        )

        # Top genres
        genre_items = list(stats['top_genres'].items())[:10]
        genre_fig = px.bar(
            x=[item[1] for item in genre_items],
            y=[item[0] for item in genre_items],
            orientation='h',
            title="Top 10 Genres",
            labels={'x': 'Count', 'y': 'Genre'}
        )

        # Mood distribution
        mood_fig = px.pie(
            values=list(stats['mood_distribution'].values()),
            names=list(stats['mood_distribution'].keys()),
            title="Mood Distribution Across Content"
        )

        return platform_fig, subreddit_fig, genre_fig, mood_fig

    return generate_analytics

def create_cross_platform_analysis():
    """Create cross-platform relationship analysis"""

    def analyze_cross_platform():
        # Theme overlap heatmap
        platforms = list(service.platform_index.keys())
        theme_overlap = {}

        for p1 in platforms:
            theme_overlap[p1] = {}
            p1_themes = set()
            for item in service.platform_index[p1]:
                p1_themes.update(item['themes'])

            for p2 in platforms:
                p2_themes = set()
                for item in service.platform_index[p2]:
                    p2_themes.update(item['themes'])

                if p1_themes and p2_themes:
                    overlap = len(p1_themes & p2_themes)
                    total = len(p1_themes | p2_themes)
                    theme_overlap[p1][p2] = overlap / max(total, 1)
                else:
                    theme_overlap[p1][p2] = 0

        platforms_list = list(platforms)
        correlation_matrix = [[theme_overlap[p1][p2] for p2 in platforms_list] for p1 in platforms_list]

        heatmap_fig = px.imshow(
            correlation_matrix,
            x=platforms_list,
            y=platforms_list,
            title="Cross-Platform Theme Correlation Matrix",
            color_continuous_scale="viridis",
            aspect="auto"
        )

        # Network analysis
        network_data = []
        for item in service.content_database:
            network_data.append({
                'title': item['title'],
                'platform': item['platform'],
                'popularity': item['popularity_score'],
                'themes': ', '.join(item['themes'][:3]),
                'x': np.random.uniform(-1, 1),
                'y': np.random.uniform(-1, 1)
            })

        network_df = pd.DataFrame(network_data)

        color_map = {'anime': 'red', 'game': 'blue', 'movie': 'green'}
        colors = [color_map.get(platform, 'gray') for platform in network_df['platform']]

        network_fig = px.scatter(
            network_df,
            x='x', y='y',
            color='platform',
            size='popularity',
            hover_data=['title', 'themes'],
            title="Content Network by Platform",
            labels={'color': 'Platform'},
            color_discrete_map=color_map
        )

        network_fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))

        return heatmap_fig, network_fig

    return analyze_cross_platform

def create_market_opportunities():
    """Create market opportunity analysis"""

    def analyze_opportunities():
        # Get theme distribution by platform
        platform_themes = {}
        for platform, items in service.platform_index.items():
            theme_counts = Counter()
            for item in items:
                for theme in item['themes']:
                    theme_counts[theme] += 1
            platform_themes[platform] = theme_counts

        # Find content gaps
        all_themes = set()
        for themes in platform_themes.values():
            all_themes.update(themes.keys())

        opportunity_data = []
        platforms = list(service.platform_index.keys())

        for theme in all_themes:
            theme_data = {'theme': theme}
            counts = []
            for platform in platforms:
                count = platform_themes[platform].get(theme, 0)
                theme_data[platform] = count
                counts.append(count)

            if max(counts) > 0:
                # Calculate opportunity score
                variance = np.var(counts)
                mean_count = np.mean(counts)
                opportunity_score = variance / max(mean_count, 1)
                theme_data['opportunity_score'] = opportunity_score
                opportunity_data.append(theme_data)

        # Sort and get top opportunities
        opportunity_data.sort(key=lambda x: x['opportunity_score'], reverse=True)
        top_opportunities = opportunity_data[:10]

        # Create heatmap
        themes = [item['theme'] for item in top_opportunities]
        heatmap_data = []
        for theme_item in top_opportunities:
            heatmap_data.append([theme_item.get(platform, 0) for platform in platforms])

        opportunity_fig = px.imshow(
            heatmap_data,
            x=platforms,
            y=themes,
            title="Market Opportunity Heatmap - Content Gaps",
            labels=dict(color="Content Count"),
            color_continuous_scale="RdYlBu_r"
        )

        # Generate opportunity text
        ranking_text = "TOP MARKET OPPORTUNITIES:\n" + "="*60 + "\n\n"
        for i, item in enumerate(top_opportunities[:5], 1):
            theme = item['theme'].replace('_', ' ').title()
            ranking_text += f"{i}. {theme}\n"
            ranking_text += f"   Opportunity Score: {item['opportunity_score']:.2f}\n"

            platform_counts = {p: item.get(p, 0) for p in platforms}
            max_platform = max(platform_counts, key=platform_counts.get)
            min_platform = min(platform_counts, key=platform_counts.get)

            ranking_text += f"   Strong in: {max_platform} ({platform_counts[max_platform]} items)\n"
            ranking_text += f"   Weak in: {min_platform} ({platform_counts[min_platform]} items)\n"
            ranking_text += f"   Strategy: Develop {theme.lower()} content for {min_platform} platform\n\n"

        return opportunity_fig, ranking_text

    return analyze_opportunities

def create_search_interface():
    """Create advanced search interface"""

    def search_content_advanced(query, platform_filter, subreddit_filter, limit):
        if not query.strip():
            return "Please enter a search query", None

        results = service.search_content(
            query.strip(),
            platform=platform_filter if platform_filter != "All" else None,
            subreddit=subreddit_filter if subreddit_filter != "All" else None,
            limit=int(limit)
        )

        if not results:
            return "No results found. Try different search terms or filters.", None

        # Format results
        results_text = f"SEARCH RESULTS FOR: '{query}'\n"
        results_text += f"Filters: Platform={platform_filter}, Subreddit={subreddit_filter}\n"
        results_text += "="*70 + "\n\n"

        for i, result in enumerate(results, 1):
            results_text += f"{i:2d}. {result['title']} ({result['platform'].upper()})\n"
            results_text += f"    Match Score: {result['match_score']} | r/{result.get('subreddit', 'unknown')}\n"
            results_text += f"    Genres: {', '.join(result['genres'][:3])}\n"
            results_text += f"    Themes: {', '.join(result['themes'][:3])}\n"
            results_text += f"    {result['description'][:80]}{'...' if len(result['description']) > 80 else ''}\n\n"

        # Create visualization
        platforms = [r['platform'] for r in results]
        match_scores = [r['match_score'] for r in results]
        titles = [r['title'][:20] + "..." if len(r['title']) > 20 else r['title'] for r in results]

        search_fig = px.bar(
            x=match_scores,
            y=titles,
            orientation='h',
            title=f"Search Results for '{query}'",
            labels={'x': 'Match Score', 'y': 'Content'},
            color=platforms,
            color_discrete_sequence=px.colors.qualitative.Set3
        )

        return results_text, search_fig

    return search_content_advanced

# Create the complete dashboard
def create_dashboard():
    """Create the comprehensive dashboard"""

    with gr.Blocks(
        title="Cross-Platform Entertainment Recommendation System",
        theme=gr.themes.Soft(),
        css="""
        .gradio-container {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .tab-nav {
            background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        }
        """
    ) as demo:

        gr.Markdown("""
        # Cross-Platform Entertainment Recommendation System

        **Discover your next favorite content across anime, games, and movies!**

        Based on Reddit discussions, MyAnimeList data, TMDB movies, and Twitch gaming trends.
        Powered by content DNA profiling and collaborative filtering.
        """)

        with gr.Tabs():
            # Main Recommendation Tab
            with gr.TabItem("Get Recommendations"):
                gr.Markdown("### Find Similar Content Across Platforms")

                with gr.Row():
                    with gr.Column(scale=1):
                        title_input = gr.Textbox(
                            label="Enter a title you enjoyed",
                            placeholder="e.g., Attack on Titan, The Last of Us, Mad Max: Fury Road",
                            value=""
                        )

                        with gr.Row():
                            num_recs = gr.Slider(
                                minimum=1, maximum=15, value=8, step=1,
                                label="Number of recommendations"
                            )
                            platform_filter = gr.Dropdown(
                                choices=["All", "Anime", "Game", "Movie"],
                                value="All",
                                label="Platform Filter"
                            )

                        mood_filter = gr.Dropdown(
                            choices=["All", "Dark", "Intense", "Emotional", "Light_hearted", "Romantic", "Mysterious", "Philosophical"],
                            value="All",
                            label="Mood Filter"
                        )

                        recommend_btn = gr.Button("Get Recommendations", variant="primary", size="lg")

                    with gr.Column(scale=2):
                        recommendations_output = gr.Textbox(
                            label="Recommendations",
                            lines=25,
                            max_lines=30,
                            show_copy_button=True
                        )

                with gr.Row():
                    with gr.Column():
                        platform_chart = gr.Plot(label="Platform Distribution")
                        summary_stats = gr.Textbox(
                            label="Summary Statistics",
                            lines=8,
                            interactive=False
                        )

                    with gr.Column():
                        confidence_chart = gr.Plot(label="Confidence Scores")

                recommend_btn.click(
                    create_main_recommendation_interface(),
                    inputs=[title_input, num_recs, platform_filter, mood_filter],
                    outputs=[recommendations_output, platform_chart, confidence_chart, summary_stats]
                )

            # Advanced Search Tab
            with gr.TabItem("Advanced Search"):
                gr.Markdown("### Search Content Database")

                with gr.Row():
                    with gr.Column(scale=1):
                        search_query = gr.Textbox(
                            label="Search Query",
                            placeholder="e.g., action, survival, sci-fi",
                            value=""
                        )

                        with gr.Row():
                            search_platform = gr.Dropdown(
                                choices=["All"] + list(service.platform_index.keys()),
                                value="All",
                                label="Platform"
                            )
                            search_subreddit = gr.Dropdown(
                                choices=["All"] + list(service.subreddit_index.keys()),
                                value="All",
                                label="Subreddit"
                            )

                        search_limit = gr.Slider(
                            minimum=5, maximum=50, value=15, step=5,
                            label="Max Results"
                        )

                        search_btn = gr.Button("Search", variant="primary")

                    with gr.Column(scale=2):
                        search_results = gr.Textbox(
                            label="Search Results",
                            lines=20,
                            show_copy_button=True
                        )

                search_visualization = gr.Plot(label="Search Results Visualization")

                search_btn.click(
                    create_search_interface(),
                    inputs=[search_query, search_platform, search_subreddit, search_limit],
                    outputs=[search_results, search_visualization]
                )

            # Analytics Dashboard Tab
            with gr.TabItem("Analytics Dashboard"):
                gr.Markdown("### Database Statistics and Insights")

                analytics_btn = gr.Button("Generate Analytics", variant="primary", size="lg")

                with gr.Row():
                    platform_dist_chart = gr.Plot(label="Platform Distribution")
                    subreddit_dist_chart = gr.Plot(label="Subreddit Distribution")

                with gr.Row():
                    top_genres_chart = gr.Plot(label="Top Genres")
                    mood_dist_chart = gr.Plot(label="Mood Distribution")

                analytics_btn.click(
                    create_analytics_dashboard(),
                    outputs=[platform_dist_chart, subreddit_dist_chart, top_genres_chart, mood_dist_chart]
                )

            # Cross-Platform Analysis Tab
            with gr.TabItem("Cross-Platform Analysis"):
                gr.Markdown("### Understanding Content Relationships Across Platforms")

                cross_platform_btn = gr.Button("Analyze Cross-Platform Patterns", variant="primary", size="lg")

                correlation_heatmap = gr.Plot(label="Theme Correlation Matrix")
                network_graph = gr.Plot(label="Content Network Visualization")

                cross_platform_btn.click(
                    create_cross_platform_analysis(),
                    outputs=[correlation_heatmap, network_graph]
                )

            # Market Opportunities Tab
            with gr.TabItem("Market Opportunities"):
                gr.Markdown("### Identify Content Gaps and Business Opportunities")

                opportunities_btn = gr.Button("Analyze Market Gaps", variant="primary", size="lg")

                with gr.Row():
                    with gr.Column(scale=2):
                        opportunity_heatmap = gr.Plot(label="Content Gap Analysis")

                    with gr.Column(scale=1):
                        opportunity_ranking = gr.Textbox(
                            label="Top Market Opportunities",
                            lines=20,
                            show_copy_button=True,
                            interactive=False
                        )

                opportunities_btn.click(
                    create_market_opportunities(),
                    outputs=[opportunity_heatmap, opportunity_ranking]
                )

            # System Status Tab
            with gr.TabItem("System Status"):
                gr.Markdown("### Data Sources and System Information")

                with gr.Row():
                    with gr.Column():
                        gr.Markdown("#### Active Data Sources")
                        data_sources = gr.Textbox(
                            value=f"""Current Data Sources:
• Reddit API: {', '.join([f'r/{sub}' for subs in TARGET_SUBREDDITS.values() for sub in subs])}
• Google Sheets: {GOOGLE_SHEETS_CONFIG['SHEET_ID']}
• TMDB API: Movie metadata
• Twitch API: Gaming trends
• MyAnimeList (Jikan): Anime data

API Status:
• Reddit Client ID: {REDDIT_CONFIG['CLIENT_ID'][:10]}...
• TMDB API Key: {TMDB_CONFIG['API_KEY'][:10]}...
• Twitch Client ID: {TWITCH_CONFIG['CLIENT_ID'][:10]}...
""",
                            label="Configuration",
                            lines=12,
                            interactive=False
                        )

                        refresh_data_btn = gr.Button("Refresh Data from Sources", variant="secondary")

                    with gr.Column():
                        stats = service.get_platform_stats()
                        system_stats = gr.Textbox(
                            value=f"""System Statistics:
• Total Content Items: {stats['total_items']}
• Platforms: {', '.join(stats['platforms'].keys())}
• Active Subreddits: {len(stats['subreddits'])}
• Unique Genres: {len(stats['top_genres'])}
• Unique Themes: {len(stats['top_themes'])}
• Demographics Covered: {', '.join(stats['demographics'].keys())}

Platform Breakdown:
{chr(10).join([f'• {k.title()}: {v} items' for k, v in stats['platforms'].items()])}

Top Subreddits:
{chr(10).join([f'• r/{k}: {v} items' for k, v in list(stats['subreddits'].items())[:5]])}
""",
                            label="System Status",
                            lines=15,
                            interactive=False
                        )

                def refresh_system_stats():
                    stats = service.get_platform_stats()
                    return f"""System Statistics (Updated: {datetime.now().strftime('%H:%M:%S')}):
• Total Content Items: {stats['total_items']}
• Platforms: {', '.join(stats['platforms'].keys())}
• Active Subreddits: {len(stats['subreddits'])}
• Unique Genres: {len(stats['top_genres'])}
• Unique Themes: {len(stats['top_themes'])}
• Demographics Covered: {', '.join(stats['demographics'].keys())}

Platform Breakdown:
{chr(10).join([f'• {k.title()}: {v} items' for k, v in stats['platforms'].items()])}

Top Subreddits:
{chr(10).join([f'• r/{k}: {v} items' for k, v in list(stats['subreddits'].items())[:5]])}
"""

                refresh_data_btn.click(
                    refresh_system_stats,
                    outputs=[system_stats]
                )

        # Footer
        gr.Markdown("""
        ---
        **Cross-Platform Entertainment Recommendation System**

        Built with: Python, Gradio, Plotly, NetworkX, Pandas |
        Data Sources: Reddit API, MyAnimeList, TMDB, Twitch API, Google Sheets |
        Features: Content DNA profiling, Collaborative filtering, Cross-platform analysis, Market gap identification

        *Processing Time: ~30-40 minutes for full data collection | Expected Output: Interactive dashboard with cross-platform recommendations*
        """)

    return demo

# Launch the dashboard
if __name__ == "__main__":
    try:
        dashboard = create_dashboard()
        dashboard.launch(
            share=True,
            server_name="0.0.0.0",
            server_port=7860,
            show_api=True,
            inbrowser=True
        )
    except Exception as e:
        print(f"Error launching dashboard: {e}")
        print("Trying alternative launch configuration...")
        dashboard = create_dashboard()
        dashboard.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2db7ada7e4cd3e8b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
